# 10c. Debiased Recommender with Notebook-06 EM

**Role of this notebook.** Estimate the debiased structural recommender for validation using the same full EM structure as notebook 06, then train a serving-safe MLP to predict posterior expected utility for holdout recommendation.

**Steps.**

1. Define the serving-safe feature set for the final EU predictor.
2. Load simulated exploration rows, holdout candidates, and fixed calibrated utility variances.
3. Build the notebook-06 score proxy and initialization objects.
4. Convert simulated exploration rows into the session and interaction tables expected by the EM estimator.
5. Define the threshold solver, measurement model, E-step, phi update, theta update, and GEM acceptance logic.
6. Run full-data EM to estimate utility means, search costs, and measurement parameters.
7. Compute posterior expected utility targets for every simulated exploration interaction.
8. Train an edge MLP to predict standardized posterior EU from scalable features.
9. Score and rank holdout candidates by predicted EU.
10. Save structural estimates, EU targets, model checkpoint, holdout scores, top-100 recommendations, and metrics.

The estimator fixes the calibrated utility variances `sigma_ik` and re-estimates `mu`, `c`, and `phi` from simulated exploration behavior.


## Estimation and Leakage Rules

The EM estimator uses only information that would be observable during simulated random exploration:

| Used for estimation | Meaning |
|---|---|
| `log_wr_star` | simulated watch-ratio measurement |
| behavior heads | simulated completion, long-watch, rewatch, and negative labels for score-anchor construction |
| `belief_before_k_*` | pre-interaction session belief over categories |
| fixed `sigma_ik` | calibrated utility variance by user and category |

The estimator does not use oracle simulator objects when fitting:

| Excluded object | Why excluded |
|---|---|
| `u_star` | true validation target |
| `z_star` | latent state generated by simulator |
| true `tau_it` | simulator threshold |
| `p_accept` | simulator acceptance probability |
| holdout `u_star` | evaluation target only |

The structural model is:

$$
z_{itj}=1\{u_{ij}>\tau_{it}\},
\quad
\sum_k B_{itk}E[(U_{ik}-\tau_{it})_+] = c_i.
$$

The watch-ratio measurement equation is state-dependent:

$$
\log(WR_{itj})\mid z_{itj}=s
\sim N(m_s(x_{itj};\phi), v_s(x_{itj};\phi)).
$$


In [ ]:
from pathlib import Path
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
NEW_CODE_DIR = PROJECT_ROOT / "python_code_new"
OUTPUT_DIR = NEW_CODE_DIR / "outputs"
SIM_OUTPUT_DIR = OUTPUT_DIR / "validation_simulation"
BASE = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"
GNN_DATA_PATH = BASE / "gnn_data.pt"

EXPLORATION_PATH = SIM_OUTPUT_DIR / "validation_exploration_interactions.parquet"
HOLDOUT_PATH = SIM_OUTPUT_DIR / "validation_holdout_candidates.parquet"
MANIFEST_PATH = SIM_OUTPUT_DIR / "validation_feature_manifest.json"
STRUCTURAL_ARRAYS_PATH = OUTPUT_DIR / "structural_estimation_arrays.npz"

STRUCTURAL_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_structural_estimates.npz"
EU_TARGET_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_EU_targets.parquet"
HOLDOUT_SCORE_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_holdout_scores.parquet"
TOP100_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_top100.parquet"
METRICS_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_metrics.json"
MODEL_OUTPUT_PATH = SIM_OUTPUT_DIR / "validation_debiased_em06_EU_mlp.pt"
BASELINE_METRICS_PATH = SIM_OUTPUT_DIR / "validation_baseline_metrics.json"

RANDOM_SEED = 20260626
TRAIN_SESSION_MAX = 159
VALID_SESSION_MIN = 160
TOP_K = 100

# Notebook-06 structural controls.
WATCH_RATIO_FLOOR = 1e-3
HEAD_WEIGHT_SPEC = "naive"
NAIVE_SCORE_WEIGHTS = {"w_complete": 1.0, "w_long": 1.0, "w_rewatch": 1.0, "w_neg": -1.0}
SCORE_WEIGHTS = {"y_complete": 1.0, "y_long": 1.0, "y_rewatch": 1.0, "y_neg": -1.0}
SCORE_WEIGHT_SOURCE = "manual::naive_(1,1,1,-1)"
SCORE_SCALE = 10.0
SCORE_REFERENCE_CATEGORY_ID = -124
VAR_SHRINK_N0 = 10.0
RELIABLE_VAR_MIN_N = 10
VAR_FLOOR = 1e-6
SCALE_FLOOR = 1e-6
COST_FLOOR = 1e-6
INITIAL_SEARCH_COST_RAW = 0.5
GEM_TOL = 1e-6
THRESHOLD_RESIDUAL_TOL = 1e-5

# Notebook-10b/10c EU predictor controls.
BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 131072
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10
MIN_VALID_MSE_DELTA = 1e-5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
HIDDEN_SIZES = [256, 128, 64]
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

for p in [EXPLORATION_PATH, HOLDOUT_PATH, MANIFEST_PATH, STRUCTURAL_ARRAYS_PATH, GNN_DATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("device:", DEVICE)
print("output dir:", SIM_OUTPUT_DIR)

## 1. Serving-Safe Feature Set

The final EU-prediction MLP uses only features that can be constructed for candidate edges at recommendation time. These include static user attributes, static video attributes, context variables, and simulated pre-row history summaries.

The MLP excludes:

| Excluded group | Reason |
|---|---|
| structural estimates | avoid using fitted `mu`, `c`, `phi`, `r_hat`, or `tau_hat` as scalable serving features |
| oracle utility objects | avoid leakage from `u_star`, true utility moments, or simulator states |
| row outcomes and labels | unavailable before recommending the candidate |
| belief vectors | reserved for structural estimation rather than the serving predictor |
| `hist_author_recency_days` | no real calendar-time process in holdout ranking |

The result is a deployable edge-feature design: estimate structural targets offline, then train a model that can score new candidates from ordinary serving features.


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text())
full_continuous_cols = list(manifest["continuous_cols"])
full_binary_cols = list(manifest["binary_cols"])
full_categorical_cols = list(manifest["categorical_cols"])

DROP_FEATURES = {"hist_author_recency_days"}
FORBIDDEN_FEATURES = {
    "wr_star", "log_wr_star", "completion_star", "long_star", "rewatch_star", "neg_star",
    "watch_ratio", "play_duration", "y_complete", "y_long", "y_rewatch", "y_neg",
    "u_star", "EU", "EU_hat", "q_target", "r_hat", "tau_hat", "mu_hat", "sigma_hat", "c_hat",
    "z_star", "p_accept", "tau_it", "c_i",
}
continuous_cols = [c for c in full_continuous_cols if c not in DROP_FEATURES]
binary_cols = [c for c in full_binary_cols if c not in DROP_FEATURES]
categorical_cols = [c for c in full_categorical_cols if c not in DROP_FEATURES]
feature_cols = continuous_cols + binary_cols + categorical_cols

for c in feature_cols:
    if c in FORBIDDEN_FEATURES or c.startswith("belief_") or c.startswith("alpha_"):
        raise ValueError(f"Forbidden feature leaked into EU MLP feature set: {c}")

print("continuous features:", len(continuous_cols))
print("binary features:", len(binary_cols))
print("categorical features:", len(categorical_cols))
print("total features:", len(feature_cols))

## 2. Load Simulation Data and Fixed Variances

This section loads random-exploration rows and reshapes them into the structural estimation sample.

The key measurement is:

$$
y_{itj}=\log(WR^*_{itj}),
$$

where `WR_star` is the simulated watch ratio from notebook 10a.

The calibrated variance input is fixed during this validation estimator:

$$
\widehat\sigma_{ik}^{2}\quad\text{is loaded once and is not re-estimated.}
$$

The notebook aligns all user ids and category ids before passing arrays to EM, so each row's `user_idx_int` and `cat_idx_int` point to the correct variance and utility mean entries.


In [ ]:
struct_arrays = np.load(STRUCTURAL_ARRAYS_PATH)
sigmas_by_user = struct_arrays["sigmas_by_user"].astype(np.float64)
user_ids = struct_arrays["user_ids"].astype(np.int64)
cat_ids = struct_arrays["cat_ids"].astype(np.int64)
UTILITY_REFERENCE_CAT_IDX = int(struct_arrays["reference_cat_idx"][0])
UTILITY_REFERENCE_CATEGORY_ID = int(cat_ids[UTILITY_REFERENCE_CAT_IDX])

N_users, K_cat = sigmas_by_user.shape
uid_to_idx = {int(uid): idx for idx, uid in enumerate(user_ids)}
cat_to_idx = {int(cid): idx for idx, cid in enumerate(cat_ids)}

belief_cols = [f"belief_before_k_{k}" for k in range(K_cat)]
structural_cols = [
    "user_id", "video_id", "user_idx_int", "video_idx_int", "session_idx", "position_in_session",
    "burst_id", "session", "sess_rank", "sess_len", "date", "time", "timestamp",
    "i_cat_level1_id", "cat_idx_int", "i_video_duration",
    "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
    "u_is_lowactive_period", "hist_has_author_history", "hist_ema_watchratio",
    "wr_star", "log_wr_star", "u_star",
    "y_complete", "y_long", "y_rewatch", "y_neg",
]
exploration_cols = sorted(set(structural_cols + belief_cols + feature_cols))
holdout_cols = sorted(set(["user_id", "video_id", "u_star"] + feature_cols))

raw = pd.read_parquet(EXPLORATION_PATH, columns=exploration_cols)
holdout = pd.read_parquet(HOLDOUT_PATH, columns=holdout_cols)

df_valid = raw.copy()
df_valid["row_id"] = np.arange(len(df_valid), dtype=np.int64)
df_valid["watch_ratio"] = pd.to_numeric(df_valid["wr_star"], errors="coerce").clip(lower=WATCH_RATIO_FLOOR)
df_valid["log_watch_ratio"] = pd.to_numeric(df_valid["log_wr_star"], errors="coerce")

# Verify that validation indexes match notebook-06 arrays.
user_check = df_valid[["user_id", "user_idx_int"]].drop_duplicates()
if not np.all(user_ids[user_check["user_idx_int"].to_numpy(dtype=np.int64)] == user_check["user_id"].to_numpy(dtype=np.int64)):
    raise ValueError("user_idx_int does not align with notebook-06 user_ids")
cat_check = df_valid[["i_cat_level1_id", "cat_idx_int"]].drop_duplicates()
if not np.all(cat_ids[cat_check["cat_idx_int"].to_numpy(dtype=np.int64)] == cat_check["i_cat_level1_id"].to_numpy(dtype=np.int64)):
    raise ValueError("cat_idx_int does not align with notebook-06 cat_ids")

print("N_users:", N_users)
print("K_cat:", K_cat)
print("reference category:", UTILITY_REFERENCE_CATEGORY_ID, "idx:", UTILITY_REFERENCE_CAT_IDX)
print("exploration rows:", f"{len(df_valid):,}")
print("holdout rows:", f"{len(holdout):,}")
print("fixed sigma range:", float(sigmas_by_user.min()), float(sigmas_by_user.max()))

## 3. Score Proxy and Initialization Moments

This section builds the score proxy used for utility-mean initialization and theta anchoring. The proxy combines the four simulated behavior heads as:

$$
S_{itj}=10\{y^{complete}_{itj}+y^{long}_{itj}+y^{rewatch}_{itj}-y^{neg}_{itj}\}.
$$

The proxy is normalized within the structural estimation sample and used to initialize category-level utility moments. During theta updates, the score anchor regularizes utility means toward this behavior-derived scale without making the score proxy the final target.


In [ ]:
missing_heads = [name for name in SCORE_WEIGHTS if name not in df_valid.columns]
if missing_heads:
    raise KeyError(f"Missing score-head columns: {missing_heads}")

score_raw = np.zeros(len(df_valid), dtype=np.float64)
for name, weight in SCORE_WEIGHTS.items():
    score_raw += float(weight) * pd.to_numeric(df_valid[name], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
df_valid["score_raw"] = float(SCORE_SCALE) * score_raw

# Raw user-category score moments are used only to compute the per-user normalization scale.
raw_user_cat_moments = (
    df_valid
    .groupby(["user_id", "i_cat_level1_id"], observed=True)["score_raw"]
    .agg(
        raw_mean_score="mean",
        raw_var_score=lambda x: x.var(ddof=0),
        raw_n_score="size",
    )
    .reset_index()
)

raw_mean_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_mean_score")
)
raw_var_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_var_score")
)
raw_n_wide = (
    raw_user_cat_moments
    .pivot(index="user_id", columns="i_cat_level1_id", values="raw_n_score")
    .fillna(0.0)
)

all_users_in_score = np.sort(df_valid["user_id"].unique().astype(int))
raw_mean_wide = raw_mean_wide.reindex(index=all_users_in_score)
raw_var_wide = raw_var_wide.reindex(index=all_users_in_score)
raw_n_wide = raw_n_wide.reindex(index=all_users_in_score)

if SCORE_REFERENCE_CATEGORY_ID in raw_mean_wide.columns:
    unknown_mean_by_user = raw_mean_wide[SCORE_REFERENCE_CATEGORY_ID]
else:
    unknown_mean_by_user = pd.Series(np.nan, index=all_users_in_score, dtype=float)

global_unknown_mean = float(unknown_mean_by_user.dropna().mean()) if unknown_mean_by_user.notna().any() else float(df_valid["score_raw"].mean())
if not np.isfinite(global_unknown_mean):
    global_unknown_mean = 0.0

raw_var = raw_var_wide.to_numpy(dtype=np.float64)
raw_n = raw_n_wide.to_numpy(dtype=np.float64)
reliable_raw_var = (raw_n >= RELIABLE_VAR_MIN_N) & np.isfinite(raw_var) & (raw_var > VAR_FLOOR)

if reliable_raw_var.any():
    global_reliable_raw_sigma = float(np.sqrt(np.nanmedian(raw_var[reliable_raw_var])))
else:
    global_reliable_raw_sigma = float(np.nanstd(df_valid["score_raw"].to_numpy(dtype=np.float64)))
if not np.isfinite(global_reliable_raw_sigma) or global_reliable_raw_sigma <= SCALE_FLOOR:
    global_reliable_raw_sigma = 1.0

user_median_raw_sigma = np.full(len(all_users_in_score), global_reliable_raw_sigma, dtype=np.float64)
for i in range(len(all_users_in_score)):
    vals = raw_var[i, reliable_raw_var[i]]
    if vals.size > 0:
        user_median_raw_sigma[i] = float(np.sqrt(np.nanmedian(vals)))

unknown_scale_by_user = pd.Series(np.nan, index=all_users_in_score, dtype=float)
unknown_scale_reliable = pd.Series(False, index=all_users_in_score, dtype=bool)
if SCORE_REFERENCE_CATEGORY_ID in raw_var_wide.columns:
    unknown_var = raw_var_wide[SCORE_REFERENCE_CATEGORY_ID].to_numpy(dtype=np.float64)
    unknown_n = raw_n_wide[SCORE_REFERENCE_CATEGORY_ID].to_numpy(dtype=np.float64)
    unknown_ok = (unknown_n >= RELIABLE_VAR_MIN_N) & np.isfinite(unknown_var) & (unknown_var > VAR_FLOOR)
    unknown_scale_by_user.iloc[:] = np.where(unknown_ok, np.sqrt(unknown_var), np.nan)
    unknown_scale_reliable.iloc[:] = unknown_ok

utility_scale = unknown_scale_by_user.to_numpy(dtype=np.float64)
scale_source = np.full(len(all_users_in_score), "unknown_reliable", dtype=object)
missing_unknown_scale = ~np.isfinite(utility_scale) | (utility_scale <= SCALE_FLOOR)
utility_scale[missing_unknown_scale] = user_median_raw_sigma[missing_unknown_scale]
scale_source[missing_unknown_scale] = "user_reliable_median"
still_bad = ~np.isfinite(utility_scale) | (utility_scale <= SCALE_FLOOR)
utility_scale[still_bad] = global_reliable_raw_sigma
scale_source[still_bad] = "global_reliable_median"
utility_scale = np.maximum(utility_scale, SCALE_FLOOR)

utility_shift = unknown_mean_by_user.reindex(all_users_in_score).fillna(global_unknown_mean).to_numpy(dtype=np.float64)
shift_source = np.where(unknown_mean_by_user.reindex(all_users_in_score).notna().to_numpy(), "user_unknown_mean", "global_unknown_mean")

utility_norm_summary = pd.DataFrame({
    "user_id": all_users_in_score.astype(int),
    "utility_shift_raw_unknown_mean": utility_shift,
    "utility_scale_raw_sigma": utility_scale,
    "shift_source": shift_source,
    "scale_source": scale_source,
    "unknown_scale_reliable": unknown_scale_reliable.reindex(all_users_in_score).fillna(False).to_numpy(dtype=bool),
})

shift_map = dict(zip(utility_norm_summary["user_id"], utility_norm_summary["utility_shift_raw_unknown_mean"]))
scale_map = dict(zip(utility_norm_summary["user_id"], utility_norm_summary["utility_scale_raw_sigma"]))
df_valid["utility_shift_raw_unknown_mean"] = df_valid["user_id"].map(shift_map).astype(float)
df_valid["utility_scale_raw_sigma"] = df_valid["user_id"].map(scale_map).astype(float)
df_valid["score"] = (df_valid["score_raw"] - df_valid["utility_shift_raw_unknown_mean"]) / df_valid["utility_scale_raw_sigma"]

user_cat_score_moments = (
    df_valid
    .groupby(["user_id", "i_cat_level1_id"], observed=True)["score"]
    .agg(mean_score="mean", var_score=lambda x: x.var(ddof=0), n_score="size")
    .reset_index()
)

initial_cost_by_user = (float(INITIAL_SEARCH_COST_RAW) / utility_norm_summary["utility_scale_raw_sigma"].to_numpy(dtype=np.float64)).astype(np.float64)
initial_cost_by_user = np.maximum(initial_cost_by_user, COST_FLOOR)

print("score reference category:", SCORE_REFERENCE_CATEGORY_ID)
print("score weight source:", SCORE_WEIGHT_SOURCE)
print("users with UNKNOWN mean:", int((utility_norm_summary["shift_source"] == "user_unknown_mean").sum()))
print("users using global UNKNOWN mean fallback:", int((utility_norm_summary["shift_source"] == "global_unknown_mean").sum()))
print("scale source counts:")
print(utility_norm_summary["scale_source"].value_counts())
print("normalized score summary:")
print(df_valid["score"].describe())
print("UNKNOWN normalized mean/variance check for reliable UNKNOWN cells:")
unknown_norm = df_valid.loc[df_valid["i_cat_level1_id"].eq(SCORE_REFERENCE_CATEGORY_ID)]
if len(unknown_norm):
    display(
        unknown_norm.groupby("user_id", observed=True)["score"]
        .agg(mean="mean", var=lambda x: x.var(ddof=0), n="size")
        .describe()
    )
print("user-category score moments:", user_cat_score_moments.shape)
display(utility_norm_summary.head())
user_cat_score_moments.head()


## 4. EM Tables and Fixed Sigma

The EM functions consume two aligned tables:

| Table | One row means | Main contents |
|---|---|---|
| `session_table` | one user-session | belief vector, search cost, session id, threshold placeholder |
| `interaction_df` | one user-video interaction | user/category/session indices, log watch ratio, score proxy, and measurement features |

`sigmas_by_user` remains fixed throughout the run. The estimated parameters are:

$$
\theta = (\mu, c),
\quad
\phi = \text{watch-ratio measurement parameters}.
$$

The reference category normalization keeps the utility level identified by setting the reference category mean to zero for every user.


In [ ]:
# Align score normalization summary to notebook-06 user order and initialize c exactly as notebook 06 does.
utility_norm_summary = utility_norm_summary.set_index("user_id").reindex(user_ids).reset_index()
initial_cost_by_user = np.maximum(
    float(INITIAL_SEARCH_COST_RAW) / utility_norm_summary["utility_scale_raw_sigma"].to_numpy(dtype=np.float64),
    COST_FLOOR,
)
utility_norm_summary["initial_cost_normalized"] = initial_cost_by_user
utility_norm_summary["sigma_unknown_fixed_from_06"] = sigmas_by_user[:, UTILITY_REFERENCE_CAT_IDX]

print("fixed sigma shape:", sigmas_by_user.shape)
print("fixed UNKNOWN sigma summary:")
display(pd.Series(sigmas_by_user[:, UTILITY_REFERENCE_CAT_IDX]).describe())
print("initial cost summary:")
display(pd.Series(initial_cost_by_user).describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

In [ ]:
session_key_cols = ["user_id", "burst_id", "session"]
session_keys = df_valid[session_key_cols].drop_duplicates().copy()
session_keys["user_id"] = session_keys["user_id"].astype(int)
session_keys["burst_id"] = session_keys["burst_id"].astype(int)
session_keys["session"] = session_keys["session"].astype(int)
session_keys["user_idx"] = session_keys["user_id"].map(uid_to_idx).astype(int)

wide = session_keys.merge(
    df_valid[session_key_cols + belief_cols].drop_duplicates(subset=session_key_cols),
    on=session_key_cols,
    how="left",
    validate="one_to_one",
)
wide = wide.sort_values(["user_idx", "burst_id", "session"]).reset_index(drop=True)
wide["sess_idx"] = np.arange(len(wide), dtype=np.int64)

belief_mat = wide[belief_cols].fillna(0.0).to_numpy(dtype=np.float64)
belief_mat = np.maximum(belief_mat, 0.0)
row_sums = belief_mat.sum(axis=1, keepdims=True)
zero_belief_rows = (row_sums[:, 0] <= 1e-12)
if zero_belief_rows.any():
    belief_mat[zero_belief_rows, :] = 1.0 / K_cat
    row_sums = belief_mat.sum(axis=1, keepdims=True)
belief_mat = belief_mat / np.maximum(row_sums, 1e-12)
wide["belief_vec"] = list(belief_mat)

session_table = wide[["user_id", "user_idx", "burst_id", "session", "sess_idx", "belief_vec"]].copy()
print("estimation sessions:", session_table.shape)
print("zero-belief fallback rows:", int(zero_belief_rows.sum()))

In [ ]:
id_cols = ["row_id", "user_id", "video_id", "burst_id", "session"]
debug_time_cols = [c for c in ["date", "time", "timestamp", "session_idx", "position_in_session"] if c in df_valid.columns]
em_feature_cols = [
    "i_cat_level1_id", "i_video_duration", "sess_rank", "sess_len",
    "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
    "u_is_lowactive_period", "hist_has_author_history", "hist_ema_watchratio",
    "log_watch_ratio",
]
missing_feature_cols = [c for c in em_feature_cols if c not in df_valid.columns]
if missing_feature_cols:
    raise KeyError(f"df_valid is missing required EM columns: {missing_feature_cols}")

interaction_df = df_valid[id_cols + debug_time_cols + em_feature_cols].copy()
interaction_df["row_id"] = interaction_df["row_id"].astype(np.int64)
interaction_df["user_id"] = interaction_df["user_id"].astype(int)
interaction_df["burst_id"] = interaction_df["burst_id"].astype(int)
interaction_df["session"] = interaction_df["session"].astype(int)
interaction_df["i_cat_level1_id"] = interaction_df["i_cat_level1_id"].astype(int)

numeric_cols = [
    "i_video_duration", "sess_rank", "sess_len", "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
    "u_is_lowactive_period", "hist_has_author_history", "hist_ema_watchratio", "log_watch_ratio",
]
for col in numeric_cols:
    interaction_df[col] = pd.to_numeric(interaction_df[col], errors="coerce")

interaction_df["user_idx_int"] = interaction_df["user_id"].map(uid_to_idx)
interaction_df["cat_idx_int"] = interaction_df["i_cat_level1_id"].map(cat_to_idx)
interaction_df = interaction_df.merge(
    session_table[["user_id", "burst_id", "session", "sess_idx"]],
    on=["user_id", "burst_id", "session"],
    how="left",
    validate="many_to_one",
)

required_nonnull = numeric_cols + ["user_idx_int", "cat_idx_int", "sess_idx"]
n_before = len(interaction_df)
interaction_df = interaction_df.dropna(subset=required_nonnull).copy()
n_after = len(interaction_df)

observed_session_keys = interaction_df[["user_id", "burst_id", "session"]].drop_duplicates()
session_table = (
    session_table.drop(columns=["sess_idx"])
    .merge(observed_session_keys, on=["user_id", "burst_id", "session"], how="inner")
    .sort_values(["user_idx", "burst_id", "session"])
    .reset_index(drop=True)
)
session_table["sess_idx"] = np.arange(len(session_table), dtype=np.int64)
interaction_df = interaction_df.drop(columns=["sess_idx"]).merge(
    session_table[["user_id", "burst_id", "session", "sess_idx"]],
    on=["user_id", "burst_id", "session"],
    how="inner",
    validate="many_to_one",
)

interaction_df["user_idx_int"] = interaction_df["user_idx_int"].astype(np.int64)
interaction_df["cat_idx_int"] = interaction_df["cat_idx_int"].astype(np.int64)
interaction_df["sess_idx"] = interaction_df["sess_idx"].astype(np.int64)
interaction_df["sess_rank"] = interaction_df["sess_rank"].astype(np.float64)
interaction_df["sess_len"] = interaction_df["sess_len"].astype(np.float64)

print("interaction rows before/after non-null cleanup:", f"{n_before:,}", "->", f"{n_after:,}")
print("dropped rows:", f"{n_before - n_after:,}")

In [ ]:
obs_user_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
obs_cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
obs_sess_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)
obs_log_watch_ratio = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
session_user_idx = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
session_beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)

checks = {
    "sigmas_shape_ok": sigmas_by_user.shape == (N_users, K_cat),
    "sigmas_finite": np.isfinite(sigmas_by_user).all(),
    "sigmas_positive": (sigmas_by_user > 0).all(),
    "belief_shape_ok": session_beliefs.shape == (len(session_table), K_cat),
    "belief_rows_sum_to_one": np.allclose(session_beliefs.sum(axis=1), 1.0),
    "obs_user_idx_in_range": obs_user_idx.min() >= 0 and obs_user_idx.max() < N_users,
    "obs_cat_idx_in_range": obs_cat_idx.min() >= 0 and obs_cat_idx.max() < K_cat,
    "obs_sess_idx_in_range": obs_sess_idx.min() >= 0 and obs_sess_idx.max() < len(session_table),
    "one_session_row_per_observed_session": interaction_df[["user_id", "burst_id", "session"]].drop_duplicates().shape[0] == len(session_table),
    "no_required_missing": not interaction_df[required_nonnull].isna().any().any(),
}
for name, ok in checks.items():
    print(f"{name}: {ok}")
if not all(checks.values()):
    raise ValueError(f"Estimation input validation failed: {[name for name, ok in checks.items() if not ok]}")

## 5. EM Function Definitions

These cells define the full structural EM machinery used by this notebook:

1. Solve thresholds from belief-weighted expected marginal gains.
2. Evaluate the state-dependent watch-ratio likelihood.
3. Run the E-step to compute posterior responsibility `r_hat` and observed log likelihood.
4. Update `phi` with weighted measurement-model regressions.
5. Update `theta` with low-rank user/category utility structure, search-cost updates, and score-anchor regularization.
6. Use generalized EM acceptance rules so a theta update is rejected if it lowers the observed log likelihood beyond tolerance.

The target likelihood for an interaction is the mixture likelihood:

$$
L_{itj}
= P(z_{itj}=1\mid B_{it},\theta) f_1(y_{itj}\mid x_{itj},\phi)
+ P(z_{itj}=0\mid B_{it},\theta) f_0(y_{itj}\mid x_{itj},\phi).
$$


In [ ]:

from scipy.special import erf

threshold_diagnostic_records = []


def standard_normal_pdf(x):
    x = np.asarray(x, dtype=np.float64)
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * x * x)


def standard_normal_cdf(x):
    x = np.asarray(x, dtype=np.float64)
    return 0.5 * (1.0 + erf(x / np.sqrt(2.0)))


def theta_to_mu_matrix(theta, n_users=None, k_cat=None, center_reference=True):
    """Return the user-category utility mean matrix, shape [N_users, K_cat]."""
    if "mu" in theta:
        mu = np.asarray(theta["mu"], dtype=np.float64)
    else:
        U = np.asarray(theta["U"], dtype=np.float64)
        V = np.asarray(theta["V"], dtype=np.float64)
        inferred_n, inferred_k = U.shape[0], V.shape[0]

        b_cat = np.asarray(theta.get("b_cat", np.zeros(inferred_k)), dtype=np.float64)
        if b_cat.ndim == 0:
            b_cat = np.full(inferred_k, float(b_cat), dtype=np.float64)

        mu = b_cat[None, :] + U @ V.T

    if n_users is not None and mu.shape[0] != int(n_users):
        raise ValueError(f"mu has {mu.shape[0]} users, expected {n_users}")
    if k_cat is not None and mu.shape[1] != int(k_cat):
        raise ValueError(f"mu has {mu.shape[1]} categories, expected {k_cat}")

    if center_reference:
        reference_cat_idx = theta.get("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX", None))
        if reference_cat_idx is not None:
            reference_cat_idx = int(reference_cat_idx)
            if reference_cat_idx < 0 or reference_cat_idx >= mu.shape[1]:
                raise ValueError(f"reference_cat_idx={reference_cat_idx} is outside K={mu.shape[1]}")
            mu = mu - mu[:, [reference_cat_idx]]

    return mu


def theta_to_cost_vector(theta, n_users, cost_floor=COST_FLOOR):
    c = np.asarray(theta.get("c", np.full(n_users, float(cost_floor))), dtype=np.float64)
    if c.ndim == 0:
        c = np.full(n_users, float(c), dtype=np.float64)
    if c.shape[0] != int(n_users):
        raise ValueError(f"c has length {c.shape[0]}, expected {n_users}")
    return np.maximum(c, float(cost_floor))


def expected_marginal_gain_mixture_vec(tau, mus, sigmas, beliefs):
    """Compute E[(U - tau)+] for many belief-weighted normal mixtures."""
    tau = np.asarray(tau, dtype=np.float64)
    if tau.ndim == 1:
        tau = tau[:, None]
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-12)
    beliefs = np.asarray(beliefs, dtype=np.float64)

    a = (tau - mus) / sigmas
    tail = 1.0 - standard_normal_cdf(a)
    pdf_a = standard_normal_pdf(a)
    gain_by_cat = (mus - tau) * tail + sigmas * pdf_a
    return np.sum(beliefs * gain_by_cat, axis=1)


def summarize_threshold_residuals(residual, failed_count=0, bracket_expansions=0, label="threshold_solve"):
    residual = np.asarray(residual, dtype=np.float64).reshape(-1)
    abs_resid = np.abs(residual[np.isfinite(residual)])
    if abs_resid.size == 0:
        return {
            "label": label,
            "n_sessions": int(residual.size),
            "mean_abs_threshold_residual": np.nan,
            "median_abs_threshold_residual": np.nan,
            "p95_abs_threshold_residual": np.nan,
            "max_abs_threshold_residual": np.nan,
            "num_failed_thresholds": int(failed_count),
            "bracket_expansions": int(bracket_expansions),
        }
    return {
        "label": label,
        "n_sessions": int(residual.size),
        "mean_abs_threshold_residual": float(np.mean(abs_resid)),
        "median_abs_threshold_residual": float(np.median(abs_resid)),
        "p95_abs_threshold_residual": float(np.percentile(abs_resid, 95)),
        "max_abs_threshold_residual": float(np.max(abs_resid)),
        "num_failed_thresholds": int(failed_count),
        "bracket_expansions": int(bracket_expansions),
    }


def solve_tau_vectorized(
    mus,
    sigmas,
    beliefs,
    costs,
    max_iter=60,
    bracket_width=10.0,
    max_bracket_expansions=20,
    cost_floor=COST_FLOOR,
    return_diagnostics=False,
    diagnostic_label="threshold_solve",
    raise_on_bracket_failure=True,
):
    """Solve E[(U - tau)+] = c using validated, expanding vectorized bisection."""
    mus = np.asarray(mus, dtype=np.float64)
    sigmas = np.maximum(np.asarray(sigmas, dtype=np.float64), 1e-12)
    beliefs = np.asarray(beliefs, dtype=np.float64)
    costs = np.asarray(costs, dtype=np.float64).reshape(-1)

    if mus.shape != sigmas.shape or mus.shape != beliefs.shape:
        raise ValueError("mus, sigmas, and beliefs must have identical shape [S, K]")
    if costs.shape[0] != mus.shape[0]:
        raise ValueError("costs must have length S")

    belief_sums = beliefs.sum(axis=1, keepdims=True)
    beliefs = beliefs / np.maximum(belief_sums, 1e-12)
    costs = np.maximum(costs, float(cost_floor))

    smax = np.maximum(np.max(sigmas, axis=1), 1e-8)
    span = float(bracket_width) * smax
    lo = np.min(mus, axis=1) - span
    hi = np.max(mus, axis=1) + span

    expansions = 0
    for expansion in range(int(max_bracket_expansions) + 1):
        gain_lo = expected_marginal_gain_mixture_vec(lo, mus, sigmas, beliefs)
        gain_hi = expected_marginal_gain_mixture_vec(hi, mus, sigmas, beliefs)
        bad_lo = gain_lo < costs
        bad_hi = gain_hi > costs
        if not (bad_lo.any() or bad_hi.any()):
            break
        if expansion == int(max_bracket_expansions):
            failed = int(bad_lo.sum() + bad_hi.sum())
            diag = summarize_threshold_residuals(
                np.full_like(costs, np.nan),
                failed_count=failed,
                bracket_expansions=expansions,
                label=diagnostic_label,
            )
            msg = (
                f"Threshold bracket failed for {failed} endpoint checks after "
                f"{max_bracket_expansions} expansions. "
                f"bad_lo={int(bad_lo.sum())}, bad_hi={int(bad_hi.sum())}, "
                f"label={diagnostic_label!r}"
            )
            if raise_on_bracket_failure:
                raise ValueError(msg)
            if return_diagnostics:
                return np.full_like(costs, np.nan), diag
            raise ValueError(msg)
        expand_span = span * (2.0 ** expansion)
        lo = np.where(bad_lo, lo - expand_span, lo)
        hi = np.where(bad_hi, hi + expand_span, hi)
        expansions += 1

    for _ in range(int(max_iter)):
        mid = 0.5 * (lo + hi)
        gain = expected_marginal_gain_mixture_vec(mid, mus, sigmas, beliefs)
        should_raise_tau = gain > costs
        lo = np.where(should_raise_tau, mid, lo)
        hi = np.where(should_raise_tau, hi, mid)

    tau = 0.5 * (lo + hi)
    final_gain = expected_marginal_gain_mixture_vec(tau, mus, sigmas, beliefs)
    residual = final_gain - costs
    diag = summarize_threshold_residuals(residual, failed_count=0, bracket_expansions=expansions, label=diagnostic_label)
    if return_diagnostics:
        return tau, diag
    return tau


def solve_tau_for_sessions(session_table, theta, sigmas_by_user, max_iter=60, return_diagnostics=False, diagnostic_label="sessions"):
    """Solve one threshold per row of `session_table`."""
    mu_matrix = theta_to_mu_matrix(theta, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    cost_vector = theta_to_cost_vector(theta, n_users=sigmas_by_user.shape[0])

    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)

    session_mus = mu_matrix[sess_users]
    session_sigmas = sigmas_by_user[sess_users]
    session_costs = cost_vector[sess_users]

    return solve_tau_vectorized(
        session_mus,
        session_sigmas,
        beliefs,
        session_costs,
        max_iter=max_iter,
        return_diagnostics=return_diagnostics,
        diagnostic_label=diagnostic_label,
    )


In [ ]:

def build_watch_feature_blocks(interaction_df):
    """Build feature matrices for the watch-ratio measurement model."""
    df = interaction_df
    required = [
        "i_video_duration", "sess_rank", "sess_len",
        "ctx_hour_sin", "ctx_hour_cos", "ctx_is_weekend",
        "u_is_lowactive_period",
        "hist_has_author_history", "hist_ema_watchratio",
    ]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f"interaction_df missing watch-model columns: {missing}")

    duration = df["i_video_duration"].to_numpy(dtype=np.float32, copy=False)
    log_duration = np.log1p(np.maximum(duration, 0.0)).astype(np.float32)

    sess_rank = df["sess_rank"].to_numpy(dtype=np.float32, copy=False)
    sess_len = df["sess_len"].to_numpy(dtype=np.float32, copy=False)
    rel_rank = (sess_rank / np.maximum(sess_len, 1.0)).astype(np.float32)
    rel_rank_sq = (rel_rank * rel_rank).astype(np.float32)

    hour_sin = df["ctx_hour_sin"].to_numpy(dtype=np.float32, copy=False)
    hour_cos = df["ctx_hour_cos"].to_numpy(dtype=np.float32, copy=False)
    weekend = df["ctx_is_weekend"].to_numpy(dtype=np.float32, copy=False)
    low_activity = df["u_is_lowactive_period"].to_numpy(dtype=np.float32, copy=False)
    has_author_history = df["hist_has_author_history"].to_numpy(dtype=np.float32, copy=False)
    ema_watchratio = df["hist_ema_watchratio"].to_numpy(dtype=np.float32, copy=False)

    X_mean = np.vstack([
        log_duration,
        rel_rank,
        rel_rank_sq,
        hour_sin,
        hour_cos,
        weekend,
        low_activity,
        has_author_history,
        ema_watchratio,
    ]).T.astype(np.float32, copy=False)
    mean_names = [
        "log1p_i_video_duration",
        "sess_rank_over_sess_len",
        "sess_rank_over_sess_len_sq",
        "ctx_hour_sin",
        "ctx_hour_cos",
        "ctx_is_weekend",
        "u_is_lowactive_period",
        "hist_has_author_history",
        "hist_ema_watchratio",
    ]

    X_logvar = np.vstack([
        log_duration,
        rel_rank,
        hour_sin,
        hour_cos,
        weekend,
    ]).T.astype(np.float32, copy=False)
    logvar_names = [
        "log1p_i_video_duration",
        "sess_rank_over_sess_len",
        "ctx_hour_sin",
        "ctx_hour_cos",
        "ctx_is_weekend",
    ]

    meta = {
        "feature_names_mean": mean_names,
        "feature_names_logvar": logvar_names,
    }
    return X_mean, X_logvar, meta


def stable_log_variance_from_eta(eta, min_var=1e-4):
    eta = np.asarray(eta, dtype=np.float64)
    return np.logaddexp(eta, np.log(float(min_var)))


def init_phi_from_interactions(
    interaction_df,
    cat_ids,
    wr_col="log_watch_ratio",
    var_floor=1e-4,
    mean_sep=None,
    logvar_sep=0.20,
    jitter_mu=0.02,
    jitter_logvar=0.02,
    seed=0,
):
    """Data-informed initialization for the measurement parameters phi."""
    if wr_col not in interaction_df.columns:
        raise KeyError(f"interaction_df missing {wr_col}")

    cat_ids = np.asarray(cat_ids)
    K = len(cat_ids)
    stats = (
        interaction_df
        .groupby("i_cat_level1_id", observed=True)[wr_col]
        .agg(["mean", "var"])
        .reindex(cat_ids)
    )

    y = interaction_df[wr_col].to_numpy(dtype=np.float64, copy=False)
    global_mean = float(np.nanmean(y))
    global_var = max(float(np.nanvar(y)), float(var_floor))
    global_std = float(np.sqrt(global_var))
    if mean_sep is None:
        mean_sep = 0.10 * global_std

    mean_k = stats["mean"].to_numpy(dtype=np.float64)
    var_k = stats["var"].to_numpy(dtype=np.float64)
    mean_k = np.where(np.isfinite(mean_k), mean_k, global_mean)
    var_k = np.where(np.isfinite(var_k), var_k, global_var)
    var_k = np.maximum(var_k, float(var_floor))
    # eta is initialized near log(var - min_var), but log(var) is a safe approximation here.
    logvar_k = np.log(var_k)

    rng = np.random.default_rng(seed)
    alpha_mu0 = mean_k - 0.5 * mean_sep + rng.normal(0.0, jitter_mu, size=K)
    alpha_mu1 = mean_k + 0.5 * mean_sep + rng.normal(0.0, jitter_mu, size=K)
    alpha_logvar0 = logvar_k - 0.5 * logvar_sep + rng.normal(0.0, jitter_logvar, size=K)
    alpha_logvar1 = logvar_k + 0.5 * logvar_sep + rng.normal(0.0, jitter_logvar, size=K)

    X_mean, X_logvar, meta = build_watch_feature_blocks(interaction_df)

    return {
        "cat_ids": cat_ids.astype(int),
        "cat_id_to_idx": {int(cid): idx for idx, cid in enumerate(cat_ids)},
        "alpha_mu0": alpha_mu0.astype(np.float64),
        "alpha_mu1": alpha_mu1.astype(np.float64),
        "beta_mu0": np.zeros(X_mean.shape[1], dtype=np.float64),
        "beta_mu1": np.zeros(X_mean.shape[1], dtype=np.float64),
        "alpha_logvar0": alpha_logvar0.astype(np.float64),
        "alpha_logvar1": alpha_logvar1.astype(np.float64),
        "beta_logvar0": np.zeros(X_logvar.shape[1], dtype=np.float64),
        "beta_logvar1": np.zeros(X_logvar.shape[1], dtype=np.float64),
        "feature_names_mean": meta["feature_names_mean"],
        "feature_names_logvar": meta["feature_names_logvar"],
    }


def predict_watch_moments(interaction_df, phi, min_var=1e-4, return_logvar=False):
    """Return state-specific means and variances/log-variances: (m0, v0/logv0, m1, v1/logv1)."""
    X_mean, X_logvar, _ = build_watch_feature_blocks(interaction_df)
    cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)

    m0 = phi["alpha_mu0"][cat_idx] + X_mean @ phi["beta_mu0"]
    m1 = phi["alpha_mu1"][cat_idx] + X_mean @ phi["beta_mu1"]
    eta0 = phi["alpha_logvar0"][cat_idx] + X_logvar @ phi["beta_logvar0"]
    eta1 = phi["alpha_logvar1"][cat_idx] + X_logvar @ phi["beta_logvar1"]

    logv0 = stable_log_variance_from_eta(eta0, min_var=min_var)
    logv1 = stable_log_variance_from_eta(eta1, min_var=min_var)
    if return_logvar:
        return m0, logv0, m1, logv1
    return m0, np.exp(np.minimum(logv0, 700.0)), m1, np.exp(np.minimum(logv1, 700.0))


def log_normal_pdf_from_logvar(x, mean, log_var):
    x = np.asarray(x, dtype=np.float64)
    mean = np.asarray(mean, dtype=np.float64)
    log_var = np.asarray(log_var, dtype=np.float64)
    inv_var = np.exp(-log_var)
    return -0.5 * (np.log(2.0 * np.pi) + log_var + (x - mean) ** 2 * inv_var)


def log_normal_pdf(x, mean, var):
    var = np.maximum(np.asarray(var, dtype=np.float64), 1e-300)
    return log_normal_pdf_from_logvar(x, mean, np.log(var))


def compute_watch_loglik_given_responsibilities(interaction_df, phi, responsibilities):
    """Expected complete-data log likelihood contribution for the watch model."""
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)
    s = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, return_logvar=True)
    return float(np.sum(r * log_normal_pdf_from_logvar(s, m1, logv1) + (1.0 - r) * log_normal_pdf_from_logvar(s, m0, logv0)))


In [ ]:
from time import time

from scipy.optimize import minimize
import torch


LATENT_DIM = 15
INITIAL_SEARCH_COST = INITIAL_SEARCH_COST_RAW

# Starting regularization choices. These are intentionally mild and should be tuned by sensitivity checks.
THETA_L2_REG = 1e-5
SEARCH_COST_LOG_SHRINKAGE = 0.10
SEARCH_COST_SHRINK_TARGET = "mean"

# Anchor structural utility means to a sparse-shrunk normalized score proxy.
SCORE_ANCHOR_L2 = 0.05
SCORE_ANCHOR_N0 = 50.0
SCORE_ANCHOR_MIN_N = 1
SPARSE_SCORE_ANCHOR_MIN_WEIGHT = 0.25
UNOBSERVED_SCORE_ANCHOR_WEIGHT = 0.25


def init_theta_low_rank_from_scores(
    user_cat_score_moments,
    user_ids,
    cat_ids,
    d=8,
    initial_search_cost=0.5,
):
    """Initialize low-rank utility parameters from user-category mean score proxies."""
    score_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="mean_score")
        .reindex(index=user_ids, columns=cat_ids)
    )
    M = score_wide.to_numpy(dtype=np.float64)

    global_mean = float(np.nanmean(M))
    user_mean = np.nanmean(M, axis=1)
    cat_mean = np.nanmean(M, axis=0)
    user_mean = np.where(np.isfinite(user_mean), user_mean, global_mean)
    cat_mean = np.where(np.isfinite(cat_mean), cat_mean, global_mean)

    cat_ids_arr = np.asarray(cat_ids)
    ref_matches = np.flatnonzero(cat_ids_arr == SCORE_REFERENCE_CATEGORY_ID)
    reference_cat_idx = int(ref_matches[0]) if len(ref_matches) else int(globals().get("UTILITY_REFERENCE_CAT_IDX", 0))

    additive_fill = user_mean[:, None] + (cat_mean[None, :] - global_mean)
    M_filled = np.where(np.isfinite(M), M, additive_fill)
    M_centered = M_filled - M_filled[:, [reference_cat_idx]]

    b_cat = np.nanmean(M_centered, axis=0).astype(np.float64)
    b_cat = np.where(np.isfinite(b_cat), b_cat, 0.0)
    b_cat = b_cat - b_cat[reference_cat_idx]

    residual = M_centered - b_cat[None, :]
    residual = np.nan_to_num(residual, nan=0.0, posinf=0.0, neginf=0.0)
    residual = residual - residual.mean(axis=0, keepdims=True)

    U_svd, s_svd, Vt_svd = np.linalg.svd(residual, full_matrices=False)
    d_eff = int(min(d, len(s_svd)))
    scale = np.sqrt(np.maximum(s_svd[:d_eff], 0.0))
    U = U_svd[:, :d_eff] * scale[None, :]
    V = Vt_svd[:d_eff, :].T * scale[None, :]

    if d_eff < d:
        U = np.pad(U, ((0, 0), (0, d - d_eff)))
        V = np.pad(V, ((0, 0), (0, d - d_eff)))

    return {
        "b_cat": b_cat,
        "U": U.astype(np.float64),
        "V": V.astype(np.float64),
        "c": np.full(len(user_ids), float(initial_search_cost), dtype=np.float64),
    }

def build_score_anchor_inputs(
    user_cat_score_moments,
    user_ids,
    cat_ids,
    min_n=1,
    n0=50.0,
    sparse_min_weight=0.25,
    unobserved_weight=0.25,
):
    """Build sparse-shrunk targets and weights for score-anchored utility regularization."""
    score_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="mean_score")
        .reindex(index=user_ids, columns=cat_ids)
    )
    n_wide = (
        user_cat_score_moments
        .pivot(index="user_id", columns="i_cat_level1_id", values="n_score")
        .reindex(index=user_ids, columns=cat_ids)
        .fillna(0.0)
    )

    raw_target = score_wide.to_numpy(dtype=np.float64)
    n_obs_anchor = n_wide.to_numpy(dtype=np.float64)
    observed = np.isfinite(raw_target) & (n_obs_anchor >= float(min_n))

    reliability = n_obs_anchor / (n_obs_anchor + float(n0))
    target = np.where(observed, reliability * raw_target, 0.0)

    weights_observed = np.maximum(reliability, float(sparse_min_weight))
    weights = np.where(observed, weights_observed, float(unobserved_weight))

    return target.astype(np.float64), weights.astype(np.float64)


theta_init = init_theta_low_rank_from_scores(
    user_cat_score_moments=user_cat_score_moments,
    user_ids=user_ids,
    cat_ids=cat_ids,
    d=LATENT_DIM,
    initial_search_cost=INITIAL_SEARCH_COST,
)
theta_init["reference_cat_idx"] = UTILITY_REFERENCE_CAT_IDX
theta_init["c"] = initial_cost_by_user.copy()

score_anchor_matrix, score_anchor_weight = build_score_anchor_inputs(
    user_cat_score_moments=user_cat_score_moments,
    user_ids=user_ids,
    cat_ids=cat_ids,
    min_n=SCORE_ANCHOR_MIN_N,
    n0=SCORE_ANCHOR_N0,
    sparse_min_weight=SPARSE_SCORE_ANCHOR_MIN_WEIGHT,
    unobserved_weight=UNOBSERVED_SCORE_ANCHOR_WEIGHT,
)

phi_init = init_phi_from_interactions(interaction_df, cat_ids=cat_ids, seed=0)

theta_param_count = (
    theta_init["b_cat"].size
    + theta_init["U"].size
    + theta_init["V"].size
    + theta_init["c"].size
)
phi_param_count = sum(
    np.asarray(phi_init[name]).size
    for name in [
        "alpha_mu0", "alpha_mu1", "beta_mu0", "beta_mu1",
        "alpha_logvar0", "alpha_logvar1", "beta_logvar0", "beta_logvar1",
    ]
)

print("theta parameter count:", theta_param_count)
print("phi parameter count:", phi_param_count)
print("theta mu range:", np.percentile(theta_to_mu_matrix(theta_init), [0, 5, 50, 95, 100]))
print("initial c after per-user scale normalization:", np.percentile(theta_init["c"], [0, 50, 100]))
print("score-anchor weighted cells:", int((score_anchor_weight > 0).sum()))
print("score-anchor weight percentiles:", np.percentile(score_anchor_weight[score_anchor_weight > 0], [1, 50, 99]))
print("sparse score anchor min weight:", SPARSE_SCORE_ANCHOR_MIN_WEIGHT)
print("unobserved neutral anchor weight:", UNOBSERVED_SCORE_ANCHOR_WEIGHT)
print("reference mu check:", np.abs(theta_to_mu_matrix(theta_init)[:, UTILITY_REFERENCE_CAT_IDX]).max())

In [ ]:

def compute_e_step_with_ll(
    interaction_df,
    session_table,
    theta,
    phi,
    sigmas_by_user,
    return_session_tau=False,
    return_threshold_diagnostics=False,
    diagnostic_label="e_step",
):
    """E-step: posterior responsibilities, interaction thresholds, and observed log likelihood."""
    mu_matrix = theta_to_mu_matrix(theta, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    cost_vector = theta_to_cost_vector(theta, n_users=sigmas_by_user.shape[0])

    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
    tau_sessions, tau_diag = solve_tau_vectorized(
        mu_matrix[sess_users],
        sigmas_by_user[sess_users],
        beliefs,
        cost_vector[sess_users],
        return_diagnostics=True,
        diagnostic_label=diagnostic_label,
    )
    threshold_diagnostic_records.append(dict(tau_diag))

    if not np.isfinite(tau_sessions).all():
        bad = int((~np.isfinite(tau_sessions)).sum())
        raise RuntimeError(f"Non-finite tau_sessions in E-step: {bad} bad values; label={diagnostic_label!r}")

    max_abs_resid = tau_diag.get("max_abs_threshold_residual", np.nan)
    if np.isfinite(max_abs_resid) and max_abs_resid > THRESHOLD_RESIDUAL_TOL:
        raise RuntimeError(
            f"Threshold residual too large in E-step: max_abs_residual={max_abs_resid}, "
            f"tol={THRESHOLD_RESIDUAL_TOL}, label={diagnostic_label!r}"
        )

    u_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)
    k_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
    s_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)
    tau_i = tau_sessions[s_idx]

    mu_i = mu_matrix[u_idx, k_idx]
    sigma_i = np.maximum(sigmas_by_user[u_idx, k_idx], 1e-8)
    z_search = (mu_i - tau_i) / sigma_i

    pi_raw = standard_normal_cdf(z_search)
    if not np.isfinite(pi_raw).all():
        bad = int((~np.isfinite(pi_raw)).sum())
        raise RuntimeError(f"Non-finite pi in E-step: {bad} bad values; label={diagnostic_label!r}")
    raw_min = float(np.nanmin(pi_raw))
    raw_max = float(np.nanmax(pi_raw))
    if raw_min < -1e-12 or raw_max > 1.0 + 1e-12:
        raise RuntimeError(f"pi outside [0,1] before clipping; min={raw_min}, max={raw_max}, label={diagnostic_label!r}")

    pi = np.clip(pi_raw, 1e-12, 1.0 - 1e-12)
    if not np.isfinite(pi).all():
        bad = int((~np.isfinite(pi)).sum())
        raise RuntimeError(f"Non-finite clipped pi in E-step: {bad} bad values; label={diagnostic_label!r}")
    clipped_min = float(np.nanmin(pi))
    clipped_max = float(np.nanmax(pi))
    if clipped_min < 0.0 or clipped_max > 1.0:
        raise RuntimeError(f"pi outside [0,1] after clipping; min={clipped_min}, max={clipped_max}, label={diagnostic_label!r}")

    log_pi = np.log(pi)
    log_one_minus_pi = np.log1p(-pi)

    y = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, return_logvar=True)
    log_f0 = log_normal_pdf_from_logvar(y, m0, logv0)
    log_f1 = log_normal_pdf_from_logvar(y, m1, logv1)

    log_joint1 = log_pi + log_f1
    log_joint0 = log_one_minus_pi + log_f0
    log_lik_i = np.logaddexp(log_joint1, log_joint0)
    total_ll = float(np.sum(log_lik_i))
    if not np.isfinite(total_ll):
        raise RuntimeError(f"Non-finite observed log likelihood in E-step: ll={total_ll}; label={diagnostic_label!r}")

    r_i = np.exp(log_joint1 - log_lik_i)
    if not np.isfinite(r_i).all():
        bad = int((~np.isfinite(r_i)).sum())
        raise RuntimeError(f"Non-finite responsibilities in E-step: {bad} bad values; label={diagnostic_label!r}")
    r_min = float(np.nanmin(r_i))
    r_max = float(np.nanmax(r_i))
    if r_min < -1e-8 or r_max > 1.0 + 1e-8:
        raise RuntimeError(f"Responsibilities outside [0,1]: min={r_min}, max={r_max}, label={diagnostic_label!r}")

    outputs = [r_i, tau_i, total_ll]
    if return_session_tau:
        outputs.append(tau_sessions)
    if return_threshold_diagnostics:
        outputs.append(tau_diag)
    return tuple(outputs) if len(outputs) > 3 else (r_i, tau_i, total_ll)


r0, tau_i0, ll0, tau_s0, tau_diag0 = compute_e_step_with_ll(
    interaction_df.head(50_000),
    session_table,
    theta_init,
    phi_init,
    sigmas_by_user,
    return_session_tau=True,
    return_threshold_diagnostics=True,
    diagnostic_label="e_step_smoke",
)
print("E-step smoke responsibilities:", r0.shape, float(r0.min()), float(r0.mean()), float(r0.max()))
print("E-step smoke LL:", ll0)
print("E-step smoke threshold diagnostics:")
print(tau_diag0)


In [ ]:

from scipy.special import expit

logvar_optimizer_diagnostic_records = []


def solve_wls_with_category_fe(X, y, weights, cat_idx, k_cat, ridge=1e-8):
    """Weighted least squares with category fixed effects and dense covariates."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    weights = np.asarray(weights, dtype=np.float64).reshape(-1)
    cat_idx = np.asarray(cat_idx, dtype=np.int64).reshape(-1)

    n_obs, p = X.shape
    if n_obs == 0 or weights.sum() <= 1e-12:
        return np.zeros(k_cat, dtype=np.float64), np.zeros(p, dtype=np.float64)

    sum_wk = np.bincount(cat_idx, weights=weights, minlength=k_cat).astype(np.float64)
    sum_wk = np.maximum(sum_wk, 1e-12)
    inv_sum_wk = 1.0 / sum_wk
    d = np.bincount(cat_idx, weights=weights * y, minlength=k_cat).astype(np.float64)

    WX = X * weights[:, None]
    E = X.T @ WX
    E[np.arange(p), np.arange(p)] += float(ridge)
    f = X.T @ (weights * y)

    B = np.empty((k_cat, p), dtype=np.float64)
    for j in range(p):
        B[:, j] = np.bincount(cat_idx, weights=weights * X[:, j], minlength=k_cat)

    B_scaled = B * inv_sum_wk[:, None]
    S = E - B.T @ B_scaled
    rhs = f - B.T @ (inv_sum_wk * d)

    try:
        beta = np.linalg.solve(S, rhs)
    except np.linalg.LinAlgError:
        beta = np.linalg.lstsq(S, rhs, rcond=None)[0]
    alpha = inv_sum_wk * (d - B @ beta)
    return alpha, beta


def optimize_logvar_block(
    X_logvar,
    resid,
    weights,
    cat_idx,
    k_cat,
    alpha_init,
    beta_init,
    min_var=1e-4,
    ridge=1e-6,
    maxiter=25,
    diagnostic_label="logvar",
):
    """Optimize one state's weighted Gaussian log-variance block with stable logaddexp math."""
    X = np.asarray(X_logvar, dtype=np.float64)
    resid2 = np.asarray(resid, dtype=np.float64) ** 2
    weights = np.asarray(weights, dtype=np.float64)
    cat_idx = np.asarray(cat_idx, dtype=np.int64)
    p = X.shape[1]
    log_min_var = np.log(float(min_var))

    x0 = np.concatenate([np.asarray(alpha_init, dtype=np.float64), np.asarray(beta_init, dtype=np.float64)])

    def obj_grad(params):
        alpha = params[:k_cat]
        beta = params[k_cat:]
        eta = alpha[cat_idx] + X @ beta
        log_var = np.logaddexp(eta, log_min_var)
        inv_var = np.exp(-log_var)
        obj_terms = 0.5 * weights * (log_var + resid2 * inv_var)
        obj = float(np.sum(obj_terms) + 0.5 * float(ridge) * np.sum(params * params))

        dlogvar_deta = expit(eta - log_min_var)
        d_obj_d_eta = 0.5 * weights * dlogvar_deta * (1.0 - resid2 * inv_var)
        grad_alpha = np.bincount(cat_idx, weights=d_obj_d_eta, minlength=k_cat)
        grad_beta = X.T @ d_obj_d_eta
        grad = np.concatenate([grad_alpha, grad_beta]) + float(ridge) * params
        return obj, grad

    initial_obj, initial_grad = obj_grad(x0)
    result = minimize(
        fun=lambda p: obj_grad(p)[0],
        x0=x0,
        jac=lambda p: obj_grad(p)[1],
        method="L-BFGS-B",
        options={"maxiter": int(maxiter), "ftol": 1e-8, "maxls": 20},
    )
    final_obj, final_grad = obj_grad(result.x)
    logvar_optimizer_diagnostic_records.append({
        "label": diagnostic_label,
        "success": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "nit": int(getattr(result, "nit", -1)),
        "nfev": int(getattr(result, "nfev", -1)),
        "initial_obj": float(initial_obj),
        "final_obj": float(final_obj),
        "initial_grad_norm": float(np.linalg.norm(initial_grad)),
        "final_grad_norm": float(np.linalg.norm(final_grad)),
    })
    params = result.x
    return params[:k_cat], params[k_cat:], result


def update_phi_coordinate_descent(
    interaction_df,
    responsibilities,
    phi_old,
    n_coord_iters=1,
    min_var=1e-4,
    ridge_mean=1e-6,
    ridge_logvar=1e-6,
    logvar_maxiter=20,
):
    """M-step for phi by alternating weighted mean and log-variance updates."""
    phi = {key: (val.copy() if isinstance(val, np.ndarray) else val) for key, val in phi_old.items()}
    y = interaction_df["log_watch_ratio"].to_numpy(dtype=np.float64, copy=False)
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)
    cat_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)
    k_cat = int(len(phi["cat_ids"]))
    X_mean, X_logvar, _ = build_watch_feature_blocks(interaction_df)
    X_mean = X_mean.astype(np.float64)
    X_logvar = X_logvar.astype(np.float64)

    for coord_iter in range(int(n_coord_iters)):
        m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, min_var=min_var, return_logvar=True)
        for state, weights_state, logvar_state in [
            (0, 1.0 - r, logv0),
            (1, r, logv1),
        ]:
            w_mean = weights_state * np.exp(-logvar_state)
            alpha_mu, beta_mu = solve_wls_with_category_fe(
                X_mean, y, w_mean, cat_idx, k_cat, ridge=ridge_mean
            )
            phi[f"alpha_mu{state}"] = alpha_mu
            phi[f"beta_mu{state}"] = beta_mu

        m0, logv0, m1, logv1 = predict_watch_moments(interaction_df, phi, min_var=min_var, return_logvar=True)
        for state, weights_state, mean_state in [
            (0, 1.0 - r, m0),
            (1, r, m1),
        ]:
            alpha_lv, beta_lv, _ = optimize_logvar_block(
                X_logvar=X_logvar,
                resid=y - mean_state,
                weights=weights_state,
                cat_idx=cat_idx,
                k_cat=k_cat,
                alpha_init=phi[f"alpha_logvar{state}"],
                beta_init=phi[f"beta_logvar{state}"],
                min_var=min_var,
                ridge=ridge_logvar,
                maxiter=logvar_maxiter,
                diagnostic_label=f"coord{coord_iter}_state{state}",
            )
            phi[f"alpha_logvar{state}"] = alpha_lv
            phi[f"beta_logvar{state}"] = beta_lv

    return phi


def swap_phi_states(phi):
    """Swap the measurement-model labels z=0 and z=1."""
    out = {key: (val.copy() if isinstance(val, np.ndarray) else val) for key, val in phi.items()}
    for stem in ["alpha_mu", "beta_mu", "alpha_logvar", "beta_logvar"]:
        key0 = f"{stem}0"
        key1 = f"{stem}1"
        out[key0], out[key1] = out[key1].copy(), out[key0].copy()
    return out


def orient_phi_high_watch_state(interaction_df, phi, sample_size=200_000, seed=0):
    """Ensure z=1 is the higher predicted log-watch-ratio state."""
    if sample_size is not None and len(interaction_df) > int(sample_size):
        check_df = interaction_df.sample(n=int(sample_size), random_state=int(seed))
    else:
        check_df = interaction_df

    m0, _, m1, _ = predict_watch_moments(check_df, phi)
    mean_m0 = float(np.mean(m0))
    mean_m1 = float(np.mean(m1))
    share_m1_gt_m0 = float(np.mean(m1 > m0))
    swapped = mean_m1 < mean_m0

    if swapped:
        phi = swap_phi_states(phi)
        mean_m0, mean_m1 = mean_m1, mean_m0
        share_m1_gt_m0 = 1.0 - share_m1_gt_m0

    info = {
        "phi_state_swapped": bool(swapped),
        "phi_mean_m0": mean_m0,
        "phi_mean_m1": mean_m1,
        "phi_mean_m1_minus_m0": mean_m1 - mean_m0,
        "phi_share_m1_gt_m0": share_m1_gt_m0,
    }
    return phi, info


In [ ]:

def update_search_costs_from_thresholds(
    session_table,
    theta_without_c,
    sigmas_by_user,
    tau_sessions,
    c_floor=COST_FLOOR,
    log_shrinkage=0.10,
    shrink_target="mean",
):
    """Update c_i from the threshold equation, with optional log-space shrinkage."""
    mu_matrix = theta_to_mu_matrix(theta_without_c, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
    sess_users = session_table["user_idx"].to_numpy(dtype=np.int64, copy=False)
    beliefs = np.vstack(session_table["belief_vec"].to_numpy()).astype(np.float64)
    gains = expected_marginal_gain_mixture_vec(
        tau_sessions,
        mu_matrix[sess_users],
        sigmas_by_user[sess_users],
        beliefs,
    )
    n_users = sigmas_by_user.shape[0]
    sum_gain = np.bincount(sess_users, weights=gains, minlength=n_users)
    n_sess = np.bincount(sess_users, minlength=n_users)
    observed_users = n_sess > 0
    c_new = np.full(n_users, float(c_floor), dtype=np.float64)
    c_new[observed_users] = sum_gain[observed_users] / n_sess[observed_users]
    c_new = np.maximum(c_new, float(c_floor))

    if float(log_shrinkage) > 0.0:
        eta = float(np.clip(log_shrinkage, 0.0, 1.0))
        log_c = np.log(c_new)
        center_source = log_c[observed_users] if observed_users.any() else log_c
        if shrink_target == "median":
            center = float(np.median(center_source))
        elif shrink_target == "mean":
            center = float(np.mean(center_source))
        else:
            raise ValueError("shrink_target must be 'mean' or 'median'")
        log_c[observed_users] = (1.0 - eta) * log_c[observed_users] + eta * center
        log_c[~observed_users] = center
        c_new = np.exp(log_c)

    return np.maximum(c_new, float(c_floor))


def update_theta_lbfgs_low_rank(
    interaction_df,
    session_table,
    responsibilities,
    tau_sessions_fixed,
    theta_old,
    sigmas_by_user,
    max_iter=25,
    reg_l2=THETA_L2_REG,
    score_anchor=None,
    score_anchor_weight=None,
    score_anchor_l2=SCORE_ANCHOR_L2,
    reference_cat_idx=None,
    c_log_shrinkage=SEARCH_COST_LOG_SHRINKAGE,
    c_shrink_target=SEARCH_COST_SHRINK_TARGET,
    max_obs=None,
    seed=0,
):
    """M-step for low-rank theta using L-BFGS on weighted probit likelihood."""
    if reference_cat_idx is None:
        reference_cat_idx = theta_old.get("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX", None))
    if reference_cat_idx is None:
        raise ValueError("reference_cat_idx is required for utility-location normalization")
    reference_cat_idx = int(reference_cat_idx)

    if score_anchor is None:
        score_anchor = globals().get("score_anchor_matrix", None)
    if score_anchor_weight is None:
        score_anchor_weight = globals().get("score_anchor_weight", None)

    rng = np.random.default_rng(seed)
    n_obs = len(interaction_df)
    if max_obs is not None and int(max_obs) < n_obs:
        obs_idx = np.sort(rng.choice(n_obs, size=int(max_obs), replace=False))
    else:
        obs_idx = np.arange(n_obs)

    u_idx = interaction_df["user_idx_int"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    k_idx = interaction_df["cat_idx_int"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    sess_idx = interaction_df["sess_idx"].to_numpy(dtype=np.int64, copy=False)[obs_idx]
    r = np.asarray(responsibilities, dtype=np.float64).reshape(-1)[obs_idx]
    tau_i = np.asarray(tau_sessions_fixed, dtype=np.float64).reshape(-1)[sess_idx]
    sigma_i = np.maximum(sigmas_by_user[u_idx, k_idx], 1e-8)

    device = torch.device("cpu")
    dtype = torch.float64
    u_t = torch.tensor(u_idx, dtype=torch.long, device=device)
    k_t = torch.tensor(k_idx, dtype=torch.long, device=device)
    r_t = torch.tensor(r, dtype=dtype, device=device)
    tau_t = torch.tensor(tau_i, dtype=dtype, device=device)
    sigma_t = torch.tensor(sigma_i, dtype=dtype, device=device)

    b_cat = torch.tensor(theta_old["b_cat"], dtype=dtype, device=device, requires_grad=True)
    U = torch.tensor(theta_old["U"], dtype=dtype, device=device, requires_grad=True)
    V = torch.tensor(theta_old["V"], dtype=dtype, device=device, requires_grad=True)
    params = [b_cat, U, V]

    use_score_anchor = (
        score_anchor is not None
        and score_anchor_weight is not None
        and float(score_anchor_l2) > 0.0
    )
    if use_score_anchor:
        score_target_t = torch.tensor(np.asarray(score_anchor, dtype=np.float64), dtype=dtype, device=device)
        score_weight_t = torch.tensor(np.asarray(score_anchor_weight, dtype=np.float64), dtype=dtype, device=device)
        if score_target_t.shape != (U.shape[0], b_cat.shape[0]):
            raise ValueError("score_anchor must have shape [N_users, K_cat]")
        score_weight_sum_t = torch.clamp(torch.sum(score_weight_t), min=1.0)

    optimizer = torch.optim.LBFGS(
        params,
        max_iter=int(max_iter),
        line_search_fn="strong_wolfe",
        tolerance_grad=1e-5,
        tolerance_change=1e-7,
    )

    sqrt2 = np.sqrt(2.0)
    eps = 1e-12

    def centered_mu_for_rows(row_users, row_cats):
        raw_mu = b_cat[row_cats] + torch.sum(U[row_users] * V[row_cats], dim=1)
        ref_mu = b_cat[reference_cat_idx] + torch.sum(U[row_users] * V[reference_cat_idx], dim=1)
        return raw_mu - ref_mu

    def centered_mu_matrix_torch():
        raw = b_cat[None, :] + U @ V.T
        return raw - raw[:, [reference_cat_idx]]

    def closure():
        optimizer.zero_grad()
        mu = centered_mu_for_rows(u_t, k_t)
        z = torch.clamp((mu - tau_t) / sigma_t, -10.0, 10.0)
        pi = torch.clamp(0.5 * (1.0 + torch.erf(z / sqrt2)), eps, 1.0 - eps)
        q = r_t * torch.log(pi) + (1.0 - r_t) * torch.log(1.0 - pi)
        neg_q = -torch.mean(q)

        reg = 0.5 * float(reg_l2) * (
            torch.mean(b_cat * b_cat)
            + torch.mean(U * U)
            + torch.mean(V * V)
        )

        if use_score_anchor:
            mu_full = centered_mu_matrix_torch()
            score_resid = mu_full - score_target_t
            score_loss = torch.sum(score_weight_t * score_resid * score_resid) / score_weight_sum_t
            reg = reg + 0.5 * float(score_anchor_l2) * score_loss

        loss = neg_q + reg
        loss.backward()
        return loss

    loss_before = float(closure().detach().cpu())
    optimizer.step(closure)
    loss_after = float(closure().detach().cpu())

    theta_new = {
        "b_cat": b_cat.detach().cpu().numpy(),
        "U": U.detach().cpu().numpy(),
        "V": V.detach().cpu().numpy(),
        "reference_cat_idx": reference_cat_idx,
    }
    theta_new["c"] = update_search_costs_from_thresholds(
        session_table=session_table,
        theta_without_c=theta_new,
        sigmas_by_user=sigmas_by_user,
        tau_sessions=tau_sessions_fixed,
        c_floor=COST_FLOOR,
        log_shrinkage=c_log_shrinkage,
        shrink_target=c_shrink_target,
    )

    score_anchor_rmse = np.nan
    if use_score_anchor:
        mu_np = theta_to_mu_matrix(theta_new, n_users=sigmas_by_user.shape[0], k_cat=sigmas_by_user.shape[1])
        w_np = np.asarray(score_anchor_weight, dtype=np.float64)
        target_np = np.asarray(score_anchor, dtype=np.float64)
        denom = max(float(w_np.sum()), 1.0)
        score_anchor_rmse = float(np.sqrt(np.sum(w_np * (mu_np - target_np) ** 2) / denom))

    info = {
        "theta_lbfgs_loss_before": loss_before,
        "theta_lbfgs_loss_after": loss_after,
        "theta_lbfgs_n_obs": int(len(obs_idx)),
        "theta_l2_reg": float(reg_l2),
        "score_anchor_l2": float(score_anchor_l2),
        "score_anchor_rmse": score_anchor_rmse,
        "c_log_shrinkage": float(c_log_shrinkage),
        "c_p01": float(np.percentile(theta_new["c"], 1)),
        "c_p50": float(np.percentile(theta_new["c"], 50)),
        "c_p99": float(np.percentile(theta_new["c"], 99)),
        "reference_mu_absmax": float(np.abs(theta_to_mu_matrix(theta_new)[:, reference_cat_idx]).max()),
    }
    return theta_new, info


def run_em_low_rank(
    interaction_df,
    session_table,
    theta_init,
    phi_init,
    sigmas_by_user,
    n_em_iters=10,
    tol=1e-4,
    gem_tol=GEM_TOL,
    phi_kwargs=None,
    theta_kwargs=None,
    orient_phi=True,
    phi_orientation_kwargs=None,
    verbose=True,
    keep_best_by_ll=True,
):
    phi_kwargs = {} if phi_kwargs is None else dict(phi_kwargs)
    theta_kwargs = {} if theta_kwargs is None else dict(theta_kwargs)
    phi_orientation_kwargs = {} if phi_orientation_kwargs is None else dict(phi_orientation_kwargs)
    def copy_param_dict(params):
        return {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in params.items()}

    theta = copy_param_dict(theta_init)
    theta.setdefault("reference_cat_idx", globals().get("UTILITY_REFERENCE_CAT_IDX"))
    theta["c"] = theta_to_cost_vector(theta, sigmas_by_user.shape[0])
    phi = copy_param_dict(phi_init)

    if orient_phi:
        phi, _ = orient_phi_high_watch_state(interaction_df, phi, **phi_orientation_kwargs)

    history = []
    prev_ll = -np.inf
    best_ll = -np.inf
    best_iter = 0
    best_stage = "initial"
    best_theta = copy_param_dict(theta)
    best_phi = copy_param_dict(phi)

    for em_iter in range(1, int(n_em_iters) + 1):
        tic = time()
        best_stage_this_iter = "none"
        r_i, tau_i, ll_before, tau_sessions = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta,
            phi,
            sigmas_by_user,
            return_session_tau=True,
            diagnostic_label=f"iter{em_iter}_before_mstep",
        )
        ll_gain_from_prev = ll_before - prev_ll
        if keep_best_by_ll and float(ll_before) > best_ll:
            best_ll = float(ll_before)
            best_iter = em_iter - 1
            best_stage = "before_m_step"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi)
            best_stage_this_iter = "before_m_step"

        phi_candidate = update_phi_coordinate_descent(
            interaction_df=interaction_df,
            responsibilities=r_i,
            phi_old=phi,
            **phi_kwargs,
        )
        phi_orient_info = {}
        if orient_phi:
            phi_candidate, phi_orient_info = orient_phi_high_watch_state(
                interaction_df,
                phi_candidate,
                **phi_orientation_kwargs,
            )

        r_i, tau_i, ll_after_phi, tau_sessions = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta,
            phi_candidate,
            sigmas_by_user,
            return_session_tau=True,
            diagnostic_label=f"iter{em_iter}_after_phi",
        )
        if keep_best_by_ll and float(ll_after_phi) > best_ll:
            best_ll = float(ll_after_phi)
            best_iter = em_iter
            best_stage = "after_phi"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi_candidate)
            best_stage_this_iter = "after_phi"

        theta_before_theta = copy_param_dict(theta)
        phi = phi_candidate
        theta_candidate, theta_info = update_theta_lbfgs_low_rank(
            interaction_df=interaction_df,
            session_table=session_table,
            responsibilities=r_i,
            tau_sessions_fixed=tau_sessions,
            theta_old=theta_before_theta,
            sigmas_by_user=sigmas_by_user,
            **theta_kwargs,
        )

        _, _, ll_after_theta_candidate = compute_e_step_with_ll(
            interaction_df,
            session_table,
            theta_candidate,
            phi,
            sigmas_by_user,
            diagnostic_label=f"iter{em_iter}_after_theta_candidate",
        )
        theta_update_rejected = bool(ll_after_theta_candidate < ll_after_phi - float(gem_tol))
        theta_info = dict(theta_info) if theta_info is not None else {}
        theta_info["theta_update_rejected"] = bool(theta_update_rejected)
        if theta_update_rejected:
            theta = theta_before_theta
            ll_after_theta = float(ll_after_phi)
            theta_info["theta_info_refers_to_rejected_candidate"] = True
            theta_info["accepted_theta_ll"] = float(ll_after_phi)
            theta_info["rejected_theta_candidate_ll"] = float(ll_after_theta_candidate)
        else:
            theta = theta_candidate
            ll_after_theta = float(ll_after_theta_candidate)
            theta_info["theta_info_refers_to_rejected_candidate"] = False
            theta_info["accepted_theta_ll"] = float(ll_after_theta_candidate)
            theta_info["rejected_theta_candidate_ll"] = np.nan

        theta_ll_decreased = bool(ll_after_theta < ll_after_phi - 1e-6)
        em_ll_decreased = bool(ll_after_theta < ll_before - 1e-6)
        best_updated = False
        if keep_best_by_ll and (not theta_update_rejected) and float(ll_after_theta) > best_ll:
            best_ll = float(ll_after_theta)
            best_iter = em_iter
            best_stage = "after_theta"
            best_theta = copy_param_dict(theta)
            best_phi = copy_param_dict(phi)
            best_stage_this_iter = "after_theta"
            best_updated = True

        record = {
            "iter": em_iter,
            "ll_before": float(ll_before),
            "ll_after_phi": float(ll_after_phi),
            "ll_after_theta_candidate": float(ll_after_theta_candidate),
            "ll_after": float(ll_after_theta),
            "ll_delta_after_phi_step": float(ll_after_phi - ll_before),
            "ll_delta_after_theta_step": float(ll_after_theta - ll_after_phi),
            "ll_delta_after_mstep": float(ll_after_theta - ll_before),
            "ll_gain_from_prev": float(ll_gain_from_prev),
            "theta_update_rejected": theta_update_rejected,
            "theta_ll_decreased": theta_ll_decreased,
            "em_ll_decreased": em_ll_decreased,
            "best_ll_so_far": float(best_ll),
            "best_iter": int(best_iter),
            "best_stage_so_far": best_stage,
            "best_stage_this_iter": best_stage_this_iter,
            "is_best_iter": bool(best_updated),
            "seconds": float(time() - tic),
            **phi_orient_info,
            **theta_info,
        }
        history.append(record)
        if verbose:
            print(record)

        if em_iter > 1 and abs(ll_after_theta - prev_ll) < float(tol):
            break
        prev_ll = ll_after_theta

    if keep_best_by_ll:
        last_iter = int(history[-1]["iter"]) if history else 0
        if verbose and (best_iter != last_iter or best_stage != "after_theta"):
            print(f"Returning best observed-LL snapshot from iter {best_iter}, stage {best_stage} (best_ll={best_ll:.6f})")
        return best_theta, best_phi, pd.DataFrame(history)

    return theta, phi, pd.DataFrame(history)


## 6. Run Full-Data EM

This section estimates `mu`, `c`, and `phi` on all simulated exploration interactions. Each EM iteration performs:

$$
\text{E-step: } \widehat r_{itj}=P(z_{itj}=1\mid y_{itj},x_{itj},B_{it};\theta,\phi),
$$

$$
\text{M-step: } (\theta,\phi)\leftarrow\arg\max Q(\theta,\phi\mid \widehat r).
$$

The run uses the same full-data control settings as the structural estimator: a maximum of 200 EM iterations, likelihood-based convergence tolerance, score-anchor regularization, search-cost shrinkage, and variance/cost floors.

The saved history records log likelihood, theta acceptance, score-anchor fit, search-cost quantiles, and measurement-model diagnostics for every iteration.


In [ ]:
FULL_EM_ITERS = 200
FULL_EM_TOL = 50.0

threshold_diagnostic_records.clear()
logvar_optimizer_diagnostic_records.clear()

theta_corr, phi_corr, hist_corr = run_em_low_rank(
    interaction_df=interaction_df,
    session_table=session_table,
    theta_init={k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in theta_init.items()},
    phi_init={k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in phi_init.items()},
    sigmas_by_user=sigmas_by_user,
    n_em_iters=FULL_EM_ITERS,
    tol=FULL_EM_TOL,
    gem_tol=GEM_TOL,
    phi_kwargs={
        "n_coord_iters": 1,
        "ridge_mean": 1e-6,
        "ridge_logvar": 1e-6,
        "logvar_maxiter": 10,
    },
    theta_kwargs={
        "max_iter": 50,
        "max_obs": None,
        "reg_l2": THETA_L2_REG,
        "score_anchor_l2": SCORE_ANCHOR_L2,
        "score_anchor": score_anchor_matrix,
        "score_anchor_weight": score_anchor_weight,
        "reference_cat_idx": UTILITY_REFERENCE_CAT_IDX,
        "c_log_shrinkage": SEARCH_COST_LOG_SHRINKAGE,
        "c_shrink_target": SEARCH_COST_SHRINK_TARGET,
        "seed": 2030,
    },
    orient_phi=True,
    phi_orientation_kwargs={
        "sample_size": 200_000,
        "seed": 2030,
    },
    verbose=True,
)

hist_corr = hist_corr.copy()
hist_corr["neg_ll_before"] = -hist_corr["ll_before"]
hist_corr["neg_ll_after"] = -hist_corr["ll_after"]
hist_corr["relative_ll_gain"] = hist_corr["ll_delta_after_mstep"] / hist_corr["ll_before"].abs()

display_cols = [
    "iter", "ll_before", "ll_after_phi", "ll_after_theta_candidate", "ll_after",
    "ll_delta_after_phi_step", "ll_delta_after_theta_step", "ll_delta_after_mstep",
    "relative_ll_gain", "theta_update_rejected", "em_ll_decreased", "best_iter",
    "best_stage_so_far", "best_stage_this_iter", "seconds", "phi_state_swapped",
    "phi_mean_m0", "phi_mean_m1", "phi_mean_m1_minus_m0",
    "theta_lbfgs_loss_before", "theta_lbfgs_loss_after", "score_anchor_l2",
    "score_anchor_rmse", "reference_mu_absmax", "c_p01", "c_p50", "c_p99",
]
display(hist_corr[[col for col in display_cols if col in hist_corr.columns]])

## 7. Posterior Expected Utility Targets

After EM converges, the notebook computes posterior expected utility for every simulated exploration interaction.

For a normal utility component with threshold `tau`, the truncated means are:

$$
E[U\mid U>\tau]
= \mu + \sigma\frac{\phi(a)}{1-\Phi(a)},
\quad
E[U\mid U\le\tau]
= \mu - \sigma\frac{\phi(a)}{\Phi(a)},
\quad
 a=\frac{\tau-\mu}{\sigma}.
$$

The posterior EU target is:

$$
EU_{itj}
= \widehat r_{itj}E[u_{ij}\mid u_{ij}>\widehat\tau_{it}]
+ (1-\widehat r_{itj})E[u_{ij}\mid u_{ij}\le\widehat\tau_{it}].
$$

The standardized regression target is:

$$
q_{itj}=\frac{EU_{itj}-\widehat\mu_{ij}}{\widehat\sigma_{ij}}.
$$


In [ ]:
def truncated_normal_means(mu, sigma, tau):
    sigma = np.maximum(np.asarray(sigma, dtype=np.float64), 1e-8)
    alpha = (tau - mu) / sigma
    pdf = standard_normal_pdf(alpha)
    cdf = np.clip(standard_normal_cdf(alpha), 1e-12, 1.0 - 1e-12)
    sf = np.clip(1.0 - standard_normal_cdf(alpha), 1e-12, 1.0 - 1e-12)
    e_upper = mu + sigma * pdf / sf
    e_lower = mu - sigma * pdf / cdf
    return e_upper, e_lower

final_r_hat, final_tau_i, final_ll, final_tau_sessions = compute_e_step_with_ll(
    interaction_df=interaction_df,
    session_table=session_table,
    theta=theta_corr,
    phi=phi_corr,
    sigmas_by_user=sigmas_by_user,
    return_session_tau=True,
    diagnostic_label="validation_em06_final_saved_posterior",
)

mu_matrix = theta_to_mu_matrix(theta_corr, n_users=N_users, k_cat=K_cat)
c_vector = theta_to_cost_vector(theta_corr, n_users=N_users)
mu_i = mu_matrix[obs_user_idx, obs_cat_idx]
sigma_i = sigmas_by_user[obs_user_idx, obs_cat_idx]
e_upper, e_lower = truncated_normal_means(mu_i, sigma_i, final_tau_i)
EU = (final_r_hat * e_upper + (1.0 - final_r_hat) * e_lower).astype(np.float32)
q_target = ((EU.astype(np.float64) - mu_i) / np.maximum(sigma_i, 1e-8)).astype(np.float32)

np.savez_compressed(
    STRUCTURAL_OUTPUT_PATH,
    theta_b_cat=theta_corr["b_cat"],
    theta_U=theta_corr["U"],
    theta_V=theta_corr["V"],
    theta_c=theta_corr["c"],
    sigmas_by_user=sigmas_by_user,
    phi_alpha_mu0=phi_corr["alpha_mu0"],
    phi_alpha_mu1=phi_corr["alpha_mu1"],
    phi_beta_mu0=phi_corr["beta_mu0"],
    phi_beta_mu1=phi_corr["beta_mu1"],
    phi_alpha_logvar0=phi_corr["alpha_logvar0"],
    phi_alpha_logvar1=phi_corr["alpha_logvar1"],
    phi_beta_logvar0=phi_corr["beta_logvar0"],
    phi_beta_logvar1=phi_corr["beta_logvar1"],
    user_ids=user_ids,
    cat_ids=cat_ids,
    reference_cat_idx=np.array([UTILITY_REFERENCE_CAT_IDX], dtype=np.int64),
)

interaction_keys = interaction_df[["row_id", "user_id", "video_id", "session_idx", "position_in_session"]].copy()
source_features = df_valid[["row_id", "u_star", "wr_star", "log_wr_star"] + feature_cols]
eu_targets = interaction_keys.merge(source_features, on="row_id", how="left", validate="one_to_one")
eu_targets["EU"] = EU
eu_targets["q_target"] = q_target
eu_targets["r_hat"] = final_r_hat.astype(np.float32)
eu_targets["tau_hat"] = final_tau_i.astype(np.float32)
eu_targets["mu_hat"] = mu_i.astype(np.float32)
eu_targets["sigma_hat"] = sigma_i.astype(np.float32)
eu_targets["c_hat"] = c_vector[obs_user_idx].astype(np.float32)
eu_targets.to_parquet(EU_TARGET_OUTPUT_PATH, index=False)

print("saved structural estimates:", STRUCTURAL_OUTPUT_PATH)
print("saved EU targets:", EU_TARGET_OUTPUT_PATH)
print("final observed LL:", final_ll)
display(eu_targets[["EU", "q_target", "r_hat", "tau_hat", "mu_hat", "sigma_hat", "u_star"]].describe())
print("diagnostic corr EU with u_star:", eu_targets[["EU", "u_star"]].corr().iloc[0, 1])

## 8. Train EU Prediction MLP

The EU MLP learns a scalable approximation to posterior expected utility:

$$
\widehat q_{ij}=f_\eta(x_{ij}).
$$

Inputs are serving-safe edge features. The target is the standardized posterior EU `q_target`, so the training loss is mean squared error:

$$
\mathcal L(\eta)=\frac{1}{N}\sum_{(i,j)}(q_{ij}-f_\eta(x_{ij}))^2.
$$

Training uses early stopping on validation MSE. The model checkpoint stores the network weights, feature encoders, target normalization, and training metadata.


In [ ]:
gnn_data = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
cat_meta = gnn_data["feature_metadata"]["categorical_mappings"]

def encode_continuous(frame, cols, mean=None, std=None):
    if not cols:
        arr = np.zeros((len(frame), 0), dtype=np.float32)
        return arr, np.zeros(0, dtype=np.float32), np.ones(0, dtype=np.float32)
    arr = frame[cols].apply(pd.to_numeric, errors="coerce").astype("float32")
    if mean is None:
        mean = arr.mean(axis=0).to_numpy(dtype=np.float32)
    if std is None:
        std = arr.std(axis=0, ddof=0).replace(0, 1.0).to_numpy(dtype=np.float32)
    arr = arr.fillna(pd.Series(mean, index=cols))
    x = ((arr.to_numpy(dtype=np.float32) - mean) / np.maximum(std, 1e-6)).astype(np.float32)
    return x, mean.astype(np.float32), np.maximum(std, 1e-6).astype(np.float32)

def encode_binary(frame, cols):
    if not cols:
        return np.zeros((len(frame), 0), dtype=np.float32)
    return frame[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).clip(0, 1).astype("float32").to_numpy(dtype=np.float32)

def encode_categorical(frame, cols):
    if not cols:
        return np.zeros((len(frame), 0), dtype=np.int64), []
    encoded = []
    cardinalities = []
    for col in cols:
        info = cat_meta[col]
        mapping = info["mapping"]
        cardinalities.append(int(info["cardinality"]))
        codes = frame[col].astype("string").map(mapping).fillna(0).astype("int64").to_numpy()
        codes = np.clip(codes, 0, int(info["cardinality"]) - 1)
        encoded.append(codes)
    return np.vstack(encoded).T.astype(np.int64), cardinalities

train_mask = eu_targets["session_idx"].le(TRAIN_SESSION_MAX).to_numpy()
valid_mask = eu_targets["session_idx"].ge(VALID_SESSION_MIN).to_numpy()
train_df = eu_targets.loc[train_mask]
valid_df = eu_targets.loc[valid_mask]

x_cont_train, cont_mean, cont_std = encode_continuous(train_df, continuous_cols)
x_cont_valid, _, _ = encode_continuous(valid_df, continuous_cols, cont_mean, cont_std)
x_bin_train = encode_binary(train_df, binary_cols)
x_bin_valid = encode_binary(valid_df, binary_cols)
x_cat_train, categorical_cardinalities = encode_categorical(train_df, categorical_cols)
x_cat_valid, _ = encode_categorical(valid_df, categorical_cols)

y_train_raw = train_df["EU"].to_numpy(dtype=np.float32)
y_valid_raw = valid_df["EU"].to_numpy(dtype=np.float32)
y_mean = float(np.mean(y_train_raw))
y_std = float(np.std(y_train_raw))
if not np.isfinite(y_std) or y_std <= 1e-6:
    y_std = 1.0
y_train = ((y_train_raw - y_mean) / y_std).astype(np.float32)[:, None]
y_valid = ((y_valid_raw - y_mean) / y_std).astype(np.float32)[:, None]

print("train rows:", f"{len(train_df):,}")
print("valid rows:", f"{len(valid_df):,}")
print("target mean/std:", y_mean, y_std)

In [ ]:
class EdgeFeatureEncoder(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, emb_dim_cap=32):
        super().__init__()
        self.n_cont = int(n_cont)
        self.n_bin = int(n_bin)
        self.embeddings = nn.ModuleList()
        self.embedding_dims = []
        for cardinality in categorical_cardinalities:
            dim = min(emb_dim_cap, max(4, int(round(math.sqrt(cardinality)))))
            self.embeddings.append(nn.Embedding(int(cardinality), dim))
            self.embedding_dims.append(dim)
        self.output_dim = self.n_cont + self.n_bin + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.n_cont:
            parts.append(x_cont_batch.float())
        if self.n_bin:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=1)

class EURegressionMLP(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, hidden_sizes, dropout=0.1):
        super().__init__()
        self.encoder = EdgeFeatureEncoder(n_cont, n_bin, categorical_cardinalities)
        layers = []
        dim = self.encoder.output_dim
        for hidden in hidden_sizes:
            layers.append(nn.Linear(dim, hidden))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(dropout))
            dim = hidden
        layers.append(nn.Linear(dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        return self.mlp(self.encoder(x_cont_batch, x_bin_batch, x_cat_batch))

def make_tensor_dataset(x_cont, x_bin, x_cat, y=None):
    tensors = [torch.from_numpy(x_cont).float(), torch.from_numpy(x_bin).float(), torch.from_numpy(x_cat).long()]
    if y is not None:
        tensors.append(torch.from_numpy(y).float())
    return TensorDataset(*tensors)

train_ds = make_tensor_dataset(x_cont_train, x_bin_train, x_cat_train, y_train)
valid_ds = make_tensor_dataset(x_cont_valid, x_bin_valid, x_cat_valid, y_valid)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

model = EURegressionMLP(len(continuous_cols), len(binary_cols), categorical_cardinalities, HIDDEN_SIZES, DROPOUT).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
@torch.no_grad()
def predict_standardized(loader):
    model.eval()
    preds = []
    ys = []
    for batch in loader:
        if len(batch) == 4:
            xb_cont, xb_bin, xb_cat, yb = batch
            ys.append(yb.numpy())
        else:
            xb_cont, xb_bin, xb_cat = batch
        xb_cont = xb_cont.to(DEVICE)
        xb_bin = xb_bin.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        preds.append(model(xb_cont, xb_bin, xb_cat).detach().cpu().numpy())
    pred = np.vstack(preds).reshape(-1)
    if ys:
        return pred, np.vstack(ys).reshape(-1)
    return pred, None

def regression_metrics(pred_std, true_std, true_raw=None):
    pred_raw = pred_std * y_std + y_mean
    if true_raw is None:
        true_raw = true_std * y_std + y_mean
    return {
        "mse": float(mean_squared_error(true_raw, pred_raw)),
        "mae": float(mean_absolute_error(true_raw, pred_raw)),
        "pearson": float(pd.Series(pred_raw).corr(pd.Series(true_raw), method="pearson")),
        "spearman": float(pd.Series(pred_raw).corr(pd.Series(true_raw), method="spearman")),
    }

training_history = []
best_state = None
best_valid_mse = np.inf
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    running_n = 0
    for xb_cont, xb_bin, xb_cat, yb in train_loader:
        xb_cont = xb_cont.to(DEVICE)
        xb_bin = xb_bin.to(DEVICE)
        xb_cat = xb_cat.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.detach().cpu()) * len(yb)
        running_n += len(yb)

    valid_pred_std, valid_true_std = predict_standardized(valid_loader)
    valid_metrics = regression_metrics(valid_pred_std, valid_true_std, y_valid_raw)
    improved = valid_metrics["mse"] < (best_valid_mse - MIN_VALID_MSE_DELTA)
    if improved:
        best_valid_mse = valid_metrics["mse"]
        best_epoch = epoch
        epochs_without_improvement = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    record = {
        "epoch": epoch,
        "train_mse_standardized": running_loss / max(running_n, 1),
        "valid_mse": valid_metrics["mse"],
        "valid_mae": valid_metrics["mae"],
        "valid_pearson": valid_metrics["pearson"],
        "valid_spearman": valid_metrics["spearman"],
        "best_valid_mse": best_valid_mse,
        "best_epoch": best_epoch,
        "improved": bool(improved),
        "epochs_without_improvement": epochs_without_improvement,
    }
    training_history.append(record)
    print(record, flush=True)
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        early_stopped = True
        print(f"Early stopping at epoch {epoch}; best epoch = {best_epoch}, best validation MSE = {best_valid_mse:.6f}")
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

valid_pred_std, valid_true_std = predict_standardized(valid_loader)
final_valid_metrics = regression_metrics(valid_pred_std, valid_true_std, y_valid_raw)
valid_pred_raw = valid_pred_std * y_std + y_mean
final_valid_metrics["spearman_with_u_star_diagnostic"] = float(pd.Series(valid_pred_raw).corr(valid_df["u_star"].reset_index(drop=True), method="spearman"))
print("final validation metrics:", final_valid_metrics)

## 9. Holdout Scoring and Top-100 Ranking

The trained MLP scores each holdout candidate with predicted standardized EU. The score is converted back to the structural utility scale:

$$
\widehat{EU}_{ij}=\widehat\mu_{ij}+\widehat\sigma_{ij}\widehat q_{ij}.
$$

For each user, candidates are sorted by `EU_hat`, and the top 100 are written as the debiased recommendation list. True utility `u_star` is used only after ranking to evaluate welfare.


In [ ]:
x_cont_holdout, _, _ = encode_continuous(holdout, continuous_cols, cont_mean, cont_std)
x_bin_holdout = encode_binary(holdout, binary_cols)
x_cat_holdout, _ = encode_categorical(holdout, categorical_cols)
holdout_ds = make_tensor_dataset(x_cont_holdout, x_bin_holdout, x_cat_holdout, y=None)
holdout_loader = DataLoader(holdout_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

holdout_pred_std, _ = predict_standardized(holdout_loader)
holdout_scores = holdout[["user_id", "video_id", "u_star"]].copy()
holdout_scores["EU_hat"] = (holdout_pred_std * y_std + y_mean).astype(np.float32)
holdout_scores.to_parquet(HOLDOUT_SCORE_OUTPUT_PATH, index=False)
print("saved holdout scores:", HOLDOUT_SCORE_OUTPUT_PATH)
display(holdout_scores.head())

In [ ]:
def topk_by_score(frame, score_col, k=TOP_K):
    return (
        frame.sort_values(["user_id", score_col], ascending=[True, False], kind="mergesort")
        .groupby("user_id", group_keys=False, sort=False)
        .head(k)
        .copy()
    )

def utility_by_user(frame, method):
    out = frame.groupby("user_id", observed=True)["u_star"].mean().reset_index(name="utility_at_100")
    out["method"] = method
    return out

top100 = topk_by_score(holdout_scores, "EU_hat", TOP_K)
top100["rank"] = top100.groupby("user_id")["EU_hat"].rank(method="first", ascending=False).astype("int32")
top100 = top100[["user_id", "video_id", "rank", "EU_hat", "u_star"]].sort_values(["user_id", "rank"]).reset_index(drop=True)
top100.to_parquet(TOP100_OUTPUT_PATH, index=False)

oracle_top100 = topk_by_score(holdout_scores, "u_star", TOP_K)
rng = np.random.default_rng(RANDOM_SEED + 100)
random_top100 = (
    holdout_scores.assign(_rand=rng.random(len(holdout_scores)))
    .sort_values(["user_id", "_rand"], ascending=[True, True], kind="mergesort")
    .groupby("user_id", group_keys=False, sort=False)
    .head(TOP_K)
    .drop(columns=["_rand"])
)

per_user = pd.concat([
    utility_by_user(random_top100, "random"),
    utility_by_user(top100, "debiased_EU_mlp_em06"),
    utility_by_user(oracle_top100, "oracle"),
], ignore_index=True)

oracle_by_user = per_user.loc[per_user["method"].eq("oracle"), ["user_id", "utility_at_100"]].rename(columns={"utility_at_100": "oracle_utility_at_100"})
per_user = per_user.merge(oracle_by_user, on="user_id", how="left", validate="many_to_one")
per_user["regret_at_100"] = per_user["oracle_utility_at_100"] - per_user["utility_at_100"]

summary = (
    per_user.groupby("method", sort=False)
    .agg(
        utility_at_100=("utility_at_100", "mean"),
        regret_at_100=("regret_at_100", "mean"),
        users=("user_id", "nunique"),
        utility_sd=("utility_at_100", "std"),
    )
    .reset_index()
)
summary["se"] = summary["utility_sd"] / np.sqrt(summary["users"])
summary = summary.drop(columns=["utility_sd"])

if BASELINE_METRICS_PATH.exists():
    baseline_metrics = json.loads(BASELINE_METRICS_PATH.read_text())
    baseline_summary = pd.DataFrame(baseline_metrics.get("summary_table", []))
    if not baseline_summary.empty:
        baseline_summary = baseline_summary.loc[baseline_summary["method"].isin(["vanilla_mlp"])]
        summary = pd.concat([summary, baseline_summary], ignore_index=True, sort=False)

method_order = {"random": 0, "vanilla_mlp": 1, "debiased_EU_mlp_em06": 2, "oracle": 3}
summary = summary.sort_values("method", key=lambda s: s.map(method_order).fillna(99)).reset_index(drop=True)
print("saved top100:", TOP100_OUTPUT_PATH)
display(summary)

## 10. Save Debiased EM06 Artifacts

This section writes the complete output bundle:

| Output | Contents |
|---|---|
| `validation_debiased_em06_structural_estimates.npz` | estimated `theta`, `phi`, fixed `sigma`, ids, and EM history |
| `validation_debiased_em06_EU_targets.parquet` | posterior EU and standardized EU targets for exploration rows |
| `validation_debiased_em06_EU_mlp.pt` | trained EU prediction model and preprocessing metadata |
| `validation_debiased_em06_holdout_scores.parquet` | one row per holdout candidate with `EU_hat` |
| `validation_debiased_em06_top100.parquet` | top-100 recommendation list per user |
| `validation_debiased_em06_metrics.json` | EM diagnostics, MLP training history, and top-100 utility metrics |


In [ ]:
metrics = {
    "random_seed": RANDOM_SEED,
    "device": DEVICE,
    "em_procedure": "copied_from_notebook_06_run_em_low_rank",
    "fixed_sigma_source": str(STRUCTURAL_ARRAYS_PATH),
    "sigma_reestimated": False,
    "mu_c_phi_reestimated_from_simulated_exploration": True,
    "structural": {
        "full_em_iters": FULL_EM_ITERS,
        "full_em_tol": FULL_EM_TOL,
        "gem_tol": GEM_TOL,
        "theta_l2_reg": THETA_L2_REG,
        "score_anchor_l2": SCORE_ANCHOR_L2,
        "search_cost_log_shrinkage": SEARCH_COST_LOG_SHRINKAGE,
        "search_cost_shrink_target": SEARCH_COST_SHRINK_TARGET,
        "final_observed_log_likelihood": float(final_ll),
        "history": hist_corr.to_dict(orient="records"),
    },
    "mlp": {
        "num_epochs": NUM_EPOCHS,
        "num_epochs_run": len(training_history),
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "min_valid_mse_delta": MIN_VALID_MSE_DELTA,
        "early_stopped": bool(early_stopped),
        "best_epoch": int(best_epoch),
        "best_valid_mse": float(best_valid_mse),
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "target_mean": y_mean,
        "target_std": y_std,
        "validation_metrics": final_valid_metrics,
        "training_history": training_history,
    },
    "train_rows": int(train_mask.sum()),
    "valid_rows": int(valid_mask.sum()),
    "holdout_rows": int(len(holdout_scores)),
    "num_users": int(holdout_scores["user_id"].nunique()),
    "holdout_candidates_per_user": int(holdout_scores.groupby("user_id").size().median()),
    "top_k": TOP_K,
    "summary_table": summary.to_dict(orient="records"),
    "feature_columns_used": feature_cols,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "dropped_features": sorted(DROP_FEATURES),
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "outputs": {
        "structural_estimates": str(STRUCTURAL_OUTPUT_PATH),
        "eu_targets": str(EU_TARGET_OUTPUT_PATH),
        "holdout_scores": str(HOLDOUT_SCORE_OUTPUT_PATH),
        "top100": str(TOP100_OUTPUT_PATH),
        "metrics": str(METRICS_OUTPUT_PATH),
        "model": str(MODEL_OUTPUT_PATH),
    },
}
METRICS_OUTPUT_PATH.write_text(json.dumps(metrics, indent=2))

torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "config": {
        "continuous_cols": continuous_cols,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "categorical_cardinalities": categorical_cardinalities,
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
        "target_mean": y_mean,
        "target_std": y_std,
        "best_epoch": int(best_epoch),
        "best_valid_mse": float(best_valid_mse),
    },
    "preprocessing": {"cont_mean": cont_mean, "cont_std": cont_std},
}, MODEL_OUTPUT_PATH)

print("saved metrics:", METRICS_OUTPUT_PATH)
print("saved model:", MODEL_OUTPUT_PATH)
print(json.dumps({
    "validation_metrics": final_valid_metrics,
    "num_epochs_run": metrics["mlp"]["num_epochs_run"],
    "early_stopped": metrics["mlp"]["early_stopped"],
    "summary_table": metrics["summary_table"],
}, indent=2))